In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install psutil thop

In [ ]:
import os
import math
import copy
import random
import time
import gc
import itertools
import threading
import subprocess

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from thop import profile

In [ ]:
# heartbeat utilities

HEARTBEAT_SECS = 600

def start_heartbeat(name="task"):
    flag = {"on": True}
    start_time = time.time()

    def _hb():
        while flag["on"]:
            time.sleep(HEARTBEAT_SECS)
            if flag["on"]:
                elapsed = (time.time() - start_time) / 60.0
                print(f"[heartbeat:{name}] running... elapsed={elapsed:.2f} min", flush=True)

    t = threading.Thread(target=_hb, daemon=True)
    t.start()
    return flag, t


def stop_heartbeat(flag, thread):
    flag["on"] = False
    try:
        thread.join(timeout=1)
    except Exception:
        pass

In [ ]:
# paths

DATASET_PATH  = "/content/drive/MyDrive/298/Jena/Data/jena_climate_2009_2016.csv"
BASE_PROJ_DIR = "/content/drive/MyDrive/298/Jena/proj_v3"

BASE_MODEL_DIR    = os.path.join(BASE_PROJ_DIR, "baseModel")
BASE_STUDENTS_DIR = os.path.join(BASE_MODEL_DIR, "baseStudents")
BASE_LOGS_DIR     = os.path.join(BASE_MODEL_DIR, "logs")

FGSM_DIR         = os.path.join(BASE_PROJ_DIR, "fgsm")
FGSM_ADV_DIR     = os.path.join(FGSM_DIR, "advStudents")
FGSM_RESULTS_DIR = os.path.join(FGSM_DIR, "results")
FGSM_LOGS_DIR    = os.path.join(FGSM_RESULTS_DIR, "train_logs")

BIM_DIR          = os.path.join(BASE_PROJ_DIR, "bim")
BIM_ADV_DIR      = os.path.join(BIM_DIR, "advStudents")
BIM_RESULTS_DIR  = os.path.join(BIM_DIR, "results")
BIM_LOGS_DIR     = os.path.join(BIM_RESULTS_DIR, "train_logs")

PGD_DIR          = os.path.join(BASE_PROJ_DIR, "pgd")
PGD_ADV_DIR      = os.path.join(PGD_DIR, "advStudents")
PGD_RESULTS_DIR  = os.path.join(PGD_DIR, "results")
PGD_LOGS_DIR     = os.path.join(PGD_RESULTS_DIR, "train_logs")

for d in [
    BASE_PROJ_DIR, BASE_MODEL_DIR, BASE_STUDENTS_DIR, BASE_LOGS_DIR,
    FGSM_DIR, FGSM_ADV_DIR, FGSM_RESULTS_DIR, FGSM_LOGS_DIR,
    BIM_DIR, BIM_ADV_DIR, BIM_RESULTS_DIR, BIM_LOGS_DIR,
    PGD_DIR, PGD_ADV_DIR, PGD_RESULTS_DIR, PGD_LOGS_DIR,
]:
    os.makedirs(d, exist_ok=True)

# existing result artifacts
FGSM_RUNS_CSV = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_runs.csv")
FGSM_STATS_CSV = os.path.join(FGSM_RESULTS_DIR, "fgsm_morphence_stats.csv")
FGSM_WEIGHT_DIVERSITY_CSV = os.path.join(FGSM_RESULTS_DIR, "fgsm_weight_diversity.csv")
FGSM_PRED_DIVERSITY_CSV = os.path.join(FGSM_RESULTS_DIR, "fgsm_prediction_diversity.csv")
FGSM_XFER_MATRIX_CSV = os.path.join(FGSM_RESULTS_DIR, "fgsm_transfer_matrix.csv")

BIM_RUNS_CSV = os.path.join(BIM_RESULTS_DIR, "bim_morphence_runs.csv")
BIM_STATS_CSV = os.path.join(BIM_RESULTS_DIR, "bim_morphence_stats.csv")
BIM_WEIGHT_DIVERSITY_CSV = os.path.join(BIM_RESULTS_DIR, "bim_weight_diversity.csv")
BIM_PRED_DIVERSITY_CSV = os.path.join(BIM_RESULTS_DIR, "bim_prediction_diversity.csv")
BIM_XFER_MATRIX_CSV = os.path.join(BIM_RESULTS_DIR, "bim_transfer_matrix.csv")

PGD_RUNS_CSV = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_runs.csv")
PGD_STATS_CSV = os.path.join(PGD_RESULTS_DIR, "pgd_morphence_stats.csv")
PGD_WEIGHT_DIVERSITY_CSV = os.path.join(PGD_RESULTS_DIR, "pgd_weight_diversity.csv")
PGD_PRED_DIVERSITY_CSV = os.path.join(PGD_RESULTS_DIR, "pgd_prediction_diversity.csv")
PGD_XFER_MATRIX_CSV = os.path.join(PGD_RESULTS_DIR, "pgd_transfer_matrix.csv")

BASE_TRAIN_HISTORY_CSV = os.path.join(BASE_LOGS_DIR, "jena_base_train_history.csv")

# new computational metrics directory
COMP_METRICS_DIR = os.path.join(BASE_PROJ_DIR, "computational_metrics")
os.makedirs(COMP_METRICS_DIR, exist_ok=True)

BASE_RESOURCE_TRACE_CSV   = os.path.join(COMP_METRICS_DIR, "base_resource_trace.csv")
BASE_RESOURCE_SUMMARY_CSV = os.path.join(COMP_METRICS_DIR, "base_resource_summary.csv")
BASE_PROFILE_CSV          = os.path.join(COMP_METRICS_DIR, "base_model_profile.csv")

FGSM_TRAIN_RESOURCE_CSV = os.path.join(COMP_METRICS_DIR, "fgsm_student_train_resource_summary.csv")
BIM_TRAIN_RESOURCE_CSV  = os.path.join(COMP_METRICS_DIR, "bim_student_train_resource_summary.csv")
PGD_TRAIN_RESOURCE_CSV  = os.path.join(COMP_METRICS_DIR, "pgd_student_train_resource_summary.csv")

FGSM_EVAL_TRACE_CSV   = os.path.join(COMP_METRICS_DIR, "fgsm_eval_resource_trace.csv")
FGSM_EVAL_SUMMARY_CSV = os.path.join(COMP_METRICS_DIR, "fgsm_eval_resource_summary.csv")

BIM_EVAL_TRACE_CSV    = os.path.join(COMP_METRICS_DIR, "bim_eval_resource_trace.csv")
BIM_EVAL_SUMMARY_CSV  = os.path.join(COMP_METRICS_DIR, "bim_eval_resource_summary.csv")

PGD_EVAL_TRACE_CSV    = os.path.join(COMP_METRICS_DIR, "pgd_eval_resource_trace.csv")
PGD_EVAL_SUMMARY_CSV  = os.path.join(COMP_METRICS_DIR, "pgd_eval_resource_summary.csv")

FGSM_XFER_TRACE_CSV   = os.path.join(COMP_METRICS_DIR, "fgsm_transfer_resource_trace.csv")
FGSM_XFER_SUMMARY_CSV = os.path.join(COMP_METRICS_DIR, "fgsm_transfer_resource_summary.csv")

BIM_XFER_TRACE_CSV    = os.path.join(COMP_METRICS_DIR, "bim_transfer_resource_trace.csv")
BIM_XFER_SUMMARY_CSV  = os.path.join(COMP_METRICS_DIR, "bim_transfer_resource_summary.csv")

PGD_XFER_TRACE_CSV    = os.path.join(COMP_METRICS_DIR, "pgd_transfer_resource_trace.csv")
PGD_XFER_SUMMARY_CSV  = os.path.join(COMP_METRICS_DIR, "pgd_transfer_resource_summary.csv")

print("proj_v3 base dir:", BASE_PROJ_DIR)
print("Computational metrics dir:", COMP_METRICS_DIR)

proj_v3 base dir: /content/drive/MyDrive/298/Jena/proj_v3
Computational metrics dir: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics


In [ ]:
# global configs

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

LOOKBACK   = 24
TARGET_COL = "t_degc"
BATCH      = 64
N_STUDENTS = 4

EPS_LIST = [0.1, 0.2, 0.3, 0.5]
FGSM_TRAIN_EPS_LIST = [0.1, 0.2, 0.3, 0.5]
BIM_TRAIN_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]
PGD_TRAIN_EPS_LIST  = [0.1, 0.2, 0.3, 0.5]

BIM_ITERS = 10
PGD_ITERS = 10

FGSM_EPOCHS = 5
BIM_EPOCHS  = 5
PGD_EPOCHS  = 5

LR_BASE    = 1e-3
LR_STUDENT = 1e-3

Device: cuda


In [ ]:
# computational metrics helpers

class ResourceMonitor:
    def __init__(self, sample_interval=2.0):
        self.sample_interval = sample_interval
        self._stop_event = threading.Event()
        self._thread = None
        self.records = []
        self.max_cpu_ram_mb = 0.0
        self.max_gpu_mem_mb = 0.0

    def _query_gpu(self):
        try:
            cmd = [
                "nvidia-smi",
                "--query-gpu=utilization.gpu,memory.used,memory.total,power.draw",
                "--format=csv,noheader,nounits"
            ]
            out = subprocess.check_output(cmd).decode("utf-8").strip()
            vals = [v.strip() for v in out.split(",")]

            gpu_util = float(vals[0]) if vals[0] not in ["N/A", "[Not Supported]"] else None
            mem_used = float(vals[1]) if vals[1] not in ["N/A", "[Not Supported]"] else None
            mem_total = float(vals[2]) if vals[2] not in ["N/A", "[Not Supported]"] else None
            power_w = float(vals[3]) if vals[3] not in ["N/A", "[Not Supported]"] else None

            return {
                "gpu_util_percent": gpu_util,
                "gpu_mem_used_mb": mem_used,
                "gpu_mem_total_mb": mem_total,
                "gpu_power_w": power_w,
            }
        except Exception:
            return {
                "gpu_util_percent": None,
                "gpu_mem_used_mb": None,
                "gpu_mem_total_mb": None,
                "gpu_power_w": None,
            }

    def _sample_once(self):
        ts = time.time()

        proc = psutil.Process(os.getpid())
        cpu_ram_mb = proc.memory_info().rss / (1024 ** 2)
        self.max_cpu_ram_mb = max(self.max_cpu_ram_mb, cpu_ram_mb)

        torch_gpu_mem_mb = None
        if torch.cuda.is_available():
            try:
                torch_gpu_mem_mb = torch.cuda.memory_allocated() / (1024 ** 2)
                self.max_gpu_mem_mb = max(self.max_gpu_mem_mb, torch_gpu_mem_mb)
            except Exception:
                pass

        gpu_stats = self._query_gpu()
        if gpu_stats["gpu_mem_used_mb"] is not None:
            self.max_gpu_mem_mb = max(self.max_gpu_mem_mb, gpu_stats["gpu_mem_used_mb"])

        self.records.append({
            "timestamp": ts,
            "cpu_ram_mb": cpu_ram_mb,
            "torch_gpu_mem_mb": torch_gpu_mem_mb,
            **gpu_stats
        })

    def _run(self):
        while not self._stop_event.is_set():
            self._sample_once()
            time.sleep(self.sample_interval)

    def start(self):
        self.records = []
        self.max_cpu_ram_mb = 0.0
        self.max_gpu_mem_mb = 0.0
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._run, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread is not None:
            self._thread.join(timeout=2)

    def to_dataframe(self):
        return pd.DataFrame(self.records)

    def summary(self):
        if len(self.records) == 0:
            return {}

        dfm = pd.DataFrame(self.records).sort_values("timestamp").reset_index(drop=True)

        avg_gpu_util = None
        gpu_active_pct = None
        if dfm["gpu_util_percent"].notna().any():
            avg_gpu_util = float(dfm["gpu_util_percent"].dropna().mean())
            gpu_active_pct = float((dfm["gpu_util_percent"] > 0).mean() * 100.0)

        est_gpu_energy_j = None
        if dfm["gpu_power_w"].notna().sum() >= 2:
            t = dfm["timestamp"].values
            p = dfm["gpu_power_w"].astype(float).values
            mask = ~np.isnan(p)
            if mask.sum() >= 2:
                est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))

        return {
            "n_samples": int(len(dfm)),
            "sample_interval_sec": self.sample_interval,
            "max_cpu_ram_mb": float(self.max_cpu_ram_mb),
            "max_gpu_mem_mb": float(self.max_gpu_mem_mb),
            "avg_gpu_util_percent": avg_gpu_util,
            "gpu_active_percent_of_samples": gpu_active_pct,
            "est_gpu_energy_j": est_gpu_energy_j,
        }


def save_monitor_outputs(monitor, trace_csv, summary_csv, extra_summary=None):
    trace_df = monitor.to_dataframe()
    summary = monitor.summary()

    if extra_summary is not None:
        summary.update(extra_summary)

    trace_df.to_csv(trace_csv, index=False)
    pd.DataFrame([summary]).to_csv(summary_csv, index=False)

    print("Saved resource trace to:", trace_csv)
    print("Saved resource summary to:", summary_csv)
    print("Resource summary:", summary)

    return trace_df, summary


def append_summary_row(row, out_csv):
    if os.path.exists(out_csv):
        old = pd.read_csv(out_csv)
        old = pd.concat([old, pd.DataFrame([row])], ignore_index=True)
        old.to_csv(out_csv, index=False)
    else:
        pd.DataFrame([row]).to_csv(out_csv, index=False)


def profile_model_once(model, sample_input, out_csv, model_label="base"):
    model.eval()
    with torch.no_grad():
        macs, params = profile(model, inputs=(sample_input,), verbose=False)

    flops_est = 2 * macs

    row = {
        "model": model_label,
        "estimated_macs": float(macs),
        "estimated_flops": float(flops_est),
        "parameter_count": float(params),
    }

    pd.DataFrame([row]).to_csv(out_csv, index=False)
    print("Saved model profile to:", out_csv)
    print(row)
    return row

In [ ]:
# data prep

df = pd.read_csv(DATASET_PATH)
df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d.%m.%Y %H:%M:%S")
df = df.set_index("Date Time").sort_index()
df = df[~df.index.duplicated(keep="first")]

df = df.resample("h").ffill()
if df.head(1).isna().any(axis=None):
    df = df.bfill(limit=1)

col_map = {
    "p (mbar)": "p_mbar",
    "T (degC)": "t_degc",
    "Tpot (K)": "tpot_k",
    "rh (%)":   "rh_pct",
    "wv (m/s)": "wv_ms",
}
df_sel = df[list(col_map.keys())].rename(columns=col_map)

print("After hourly resample + ffill/bfill:")
print("Shape:", df_sel.shape)
print("Columns:", df_sel.columns.tolist())

split_idx = int(0.8 * len(df_sel))
train_df, test_df = df_sel.iloc[:split_idx].copy(), df_sel.iloc[split_idx:].copy()

scaler = MinMaxScaler()
train_scaled = pd.DataFrame(
    scaler.fit_transform(train_df),
    index=train_df.index,
    columns=train_df.columns,
)
test_scaled = pd.DataFrame(
    scaler.transform(test_df),
    index=test_df.index,
    columns=test_df.columns,
)

def make_windows(frame, lookback=24, target_col="t_degc"):
    vals = frame.values
    tgt_idx = frame.columns.get_loc(target_col)
    X, y, idx = [], [], []
    for i in range(lookback, len(vals)):
        X.append(vals[i - lookback:i])
        y.append(vals[i][tgt_idx])
        idx.append(frame.index[i])
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), np.array(idx)

X_train, y_train, t_train = make_windows(train_scaled, LOOKBACK, TARGET_COL)
X_test,  y_test,  t_test  = make_windows(test_scaled, LOOKBACK, TARGET_COL)

print("\nWindowed tensors")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test: ", X_test.shape,  "y_test: ", y_test.shape)

N_tr = len(X_train)
n_tr_core = int(0.9 * N_tr)

X_tr_core, y_tr_core = X_train[:n_tr_core], y_train[:n_tr_core]
X_val,     y_val     = X_train[n_tr_core:], y_train[n_tr_core:]

print("\nTrain/Val/Test windowed splits:")
print("  train_core:", X_tr_core.shape, y_tr_core.shape)
print("  val       :", X_val.shape, y_val.shape)
print("  test      :", X_test.shape, y_test.shape)

After hourly resample + ffill/bfill:
Shape: (70129, 5)
Columns: ['p_mbar', 't_degc', 'tpot_k', 'rh_pct', 'wv_ms']

Windowed tensors
X_train: (56079, 24, 5) y_train: (56079,)
X_test:  (14002, 24, 5) y_test:  (14002,)

Train/Val/Test windowed splits:
  train_core: (50471, 24, 5) (50471,)
  val       : (5608, 24, 5) (5608,)
  test      : (14002, 24, 5) (14002,)


In [ ]:
class TSForecastDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        return self.X[i], self.y[i]


train_dl = DataLoader(
    TSForecastDataset(X_tr_core, y_tr_core),
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
)

val_dl = DataLoader(
    TSForecastDataset(X_val, y_val),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
)

test_dl = DataLoader(
    TSForecastDataset(X_test, y_test),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
)

In [ ]:
# model

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]


class TransformerRegressor(nn.Module):
    def __init__(self, in_feats=5, d_model=128, nhead=4,
                 num_layers=4, dim_ff=256, dropout=0.1):
        super().__init__()
        self.in_proj = nn.Linear(in_feats, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.posenc  = PositionalEncoding(d_model)
        self.head    = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x):
        z = self.in_proj(x)
        z = self.posenc(z)
        z = self.encoder(z)
        zT = z[:, -1, :]
        return self.head(zT)


mse = nn.MSELoss()

def rmse_loss(pred, true):
    return torch.sqrt(mse(pred, true))

def quantile_loss(true, pred, q=0.5):
    e = true - pred
    return torch.mean(torch.maximum(q * e, (q - 1) * e))

In [ ]:
# base training with computational metrics

def train_base(model, train_dl, val_dl, epochs=20, lr=1e-3, q=0.5):
    model = model.to(device)
    opt = optim.AdamW(model.parameters(), lr=lr)
    history = []

    base_best_ckpt = os.path.join(BASE_MODEL_DIR, "jena_base_best.pth")

    hb_flag, hb_thread = start_heartbeat("base_train")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()
    best_val_loss = float("inf")

    monitor.start()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot_train = cnt_train = 0.0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                loss = rmse_loss(pred, yb) + quantile_loss(yb, pred, q)

                opt.zero_grad()
                loss.backward()
                opt.step()

                bs = xb.size(0)
                tot_train += loss.item() * bs
                cnt_train += bs

            train_loss = tot_train / max(1, cnt_train)

            model.eval()
            tot_val = cnt_val = 0.0
            with torch.no_grad():
                for xb, yb in val_dl:
                    xb, yb = xb.to(device), yb.to(device)
                    pred = model(xb)
                    v_loss = rmse_loss(pred, yb) + quantile_loss(yb, pred, q)

                    bs = xb.size(0)
                    tot_val += v_loss.item() * bs
                    cnt_val += bs

            val_loss = tot_val / max(1, cnt_val)

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0

            print(
                f"[Base] Epoch {ep:02d} | train={train_loss:.4f} | "
                f"val={val_loss:.4f} | ep_time={ep_mins:.2f} min | total={total_mins:.2f} min"
            )

            history.append({
                "epoch": ep,
                "train_loss": train_loss,
                "val_loss": val_loss,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            })

            ep_ckpt = os.path.join(BASE_MODEL_DIR, f"jena_base_epoch_{ep:02d}.pth")
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                torch.save(model.to("cpu").state_dict(), base_best_ckpt)
                model.to(device)
                print("  [Base] Updated best checkpoint (val):", base_best_ckpt)

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    df_hist = pd.DataFrame(history)
    df_hist.to_csv(BASE_TRAIN_HISTORY_CSV, index=False)
    print("Saved base train history to:", BASE_TRAIN_HISTORY_CSV)

    total_train_minutes = (time.time() - start_time) / 60.0

    _, resource_summary = save_monitor_outputs(
        monitor,
        trace_csv=BASE_RESOURCE_TRACE_CSV,
        summary_csv=BASE_RESOURCE_SUMMARY_CSV,
        extra_summary={
            "stage": "base_training",
            "total_elapsed_minutes": total_train_minutes,
            "epochs": epochs,
            "learning_rate": lr,
        }
    )

    dummy_x = torch.randn(1, LOOKBACK, X_train.shape[-1]).to(device)
    model_profile = profile_model_once(
        model=model,
        sample_input=dummy_x,
        out_csv=BASE_PROFILE_CSV,
        model_label="jena_base_transformer",
    )

    print("Final best base checkpoint (based on val):", base_best_ckpt)
    return base_best_ckpt, resource_summary, model_profile

In [ ]:
# train/load base model

base_best_ckpt = os.path.join(BASE_MODEL_DIR, "jena_base_best.pth")

if os.path.exists(base_best_ckpt):
    print("Loading existing best base model from:", base_best_ckpt)
    base_resource_summary = None
    base_model_profile = None
else:
    print("Training new base model...")
    base_init = TransformerRegressor(in_feats=X_train.shape[-1])
    base_best_ckpt, base_resource_summary, base_model_profile = train_base(
        base_init,
        train_dl=train_dl,
        val_dl=val_dl,
        epochs=20,
        lr=LR_BASE,
    )

base = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
base.load_state_dict(torch.load(base_best_ckpt, map_location=device))
base.eval()

print("Base model loaded from best checkpoint:", base_best_ckpt)

# if model already existed but profile csv doesn't, generate it once
if not os.path.exists(BASE_PROFILE_CSV):
    dummy_x = torch.randn(1, LOOKBACK, X_train.shape[-1]).to(device)
    _ = profile_model_once(
        model=base,
        sample_input=dummy_x,
        out_csv=BASE_PROFILE_CSV,
        model_label="jena_base_transformer",
    )

Training new base model...


/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[Base] Epoch 01 | train=0.0894 | val=0.0222 | ep_time=0.07 min | total=0.07 min
  [Base] Updated best checkpoint (val): /content/drive/MyDrive/298/Jena/proj_v3/baseModel/jena_base_best.pth
[Base] Epoch 02 | train=0.0321 | val=0.0262 | ep_time=0.06 min | total=0.13 min
[Base] Epoch 03 | train=0.0260 | val=0.0218 | ep_time=0.06 min | total=0.19 min
  [Base] Updated best checkpoint (val): /content/drive/MyDrive/298/Jena/proj_v3/baseModel/jena_base_best.pth
[Base] Epoch 04 | train=0.0241 | val=0.0175 | ep_time=0.06 min | total=0.25 min
  [Base] Updated best checkpoint (val): /content/drive/MyDrive/298/Jena/proj_v3/baseModel/jena_base_best.pth
[Base] Epoch 05 | train=0.0235 | val=0.0179 | ep_time=0.05 min | total=0.30 min
[Base] Epoch 06 | train=0.0225 | val=0.0218 | ep_time=0.05 min | total=0.35 min
[Base] Epoch 07 | train=0.0220 | val=0.0150 | ep_time=0.06 min | total=0.41 min
  [Base] Updated best checkpoint (val): /content/drive/MyDrive/298/Jena/proj_v3/baseModel/jena_base_best.pth
[Bas

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [ ]:
# generate base students

os.makedirs(BASE_STUDENTS_DIR, exist_ok=True)

def save_student(model_state, idx, noise_std=1e-3):
    st = TransformerRegressor(in_feats=X_train.shape[-1]).to("cpu")
    st.load_state_dict(model_state)

    with torch.no_grad():
        for _, p in st.named_parameters():
            p.add_(noise_std * torch.randn_like(p))

    out_path = os.path.join(BASE_STUDENTS_DIR, f"student_base_{idx:02d}.pth")
    torch.save(st.state_dict(), out_path)
    print("Saved base student:", out_path)
    return out_path


base_cpu = TransformerRegressor(in_feats=X_train.shape[-1]).to("cpu")
base_cpu.load_state_dict(torch.load(base_best_ckpt, map_location="cpu"))

existing_students = [f for f in os.listdir(BASE_STUDENTS_DIR) if f.endswith(".pth")]
if len(existing_students) >= N_STUDENTS:
    print(f"Found {len(existing_students)} base students, not regenerating.")
else:
    print("Generating base students...")
    for i in range(1, N_STUDENTS + 1):
        save_student(copy.deepcopy(base_cpu.state_dict()), i, noise_std=1e-3)

base_student_paths = sorted(
    os.path.join(BASE_STUDENTS_DIR, f)
    for f in os.listdir(BASE_STUDENTS_DIR)
    if f.endswith(".pth")
)

print("\nBase student paths:")
for p in base_student_paths:
    print("  ", p)

Generating base students...
Saved base student: /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_01.pth
Saved base student: /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_02.pth
Saved base student: /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_03.pth
Saved base student: /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_04.pth

Base student paths:
   /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_01.pth
   /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_02.pth
   /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_03.pth
   /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_04.pth


/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [ ]:
# generate base students

os.makedirs(BASE_STUDENTS_DIR, exist_ok=True)

def save_student(model_state, idx, noise_std=1e-3):
    st = TransformerRegressor(in_feats=X_train.shape[-1]).to("cpu")
    st.load_state_dict(model_state)

    with torch.no_grad():
        for _, p in st.named_parameters():
            p.add_(noise_std * torch.randn_like(p))

    out_path = os.path.join(BASE_STUDENTS_DIR, f"student_base_{idx:02d}.pth")
    torch.save(st.state_dict(), out_path)
    print("Saved base student:", out_path)
    return out_path


base_cpu = TransformerRegressor(in_feats=X_train.shape[-1]).to("cpu")
base_cpu.load_state_dict(torch.load(base_best_ckpt, map_location="cpu"))

existing_students = [f for f in os.listdir(BASE_STUDENTS_DIR) if f.endswith(".pth")]
if len(existing_students) >= N_STUDENTS:
    print(f"Found {len(existing_students)} base students, not regenerating.")
else:
    print("Generating base students...")
    for i in range(1, N_STUDENTS + 1):
        save_student(copy.deepcopy(base_cpu.state_dict()), i, noise_std=1e-3)

base_student_paths = sorted(
    os.path.join(BASE_STUDENTS_DIR, f)
    for f in os.listdir(BASE_STUDENTS_DIR)
    if f.endswith(".pth")
)

print("\nBase student paths:")
for p in base_student_paths:
    print("  ", p)

Found 4 base students, not regenerating.

Base student paths:
   /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_01.pth
   /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_02.pth
   /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_03.pth
   /content/drive/MyDrive/298/Jena/proj_v3/baseModel/baseStudents/student_base_04.pth


/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [ ]:
# adversarial student training with computational metrics

def adv_train_student_fgsm(student_path, out_dir, logs_dir,
                           out_suffix="_fgsm_adv",
                           epochs=5, lr=1e-3,
                           eps_list=None, alpha=0.02, q=0.5):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = FGSM_TRAIN_EPS_LIST

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"fgsm_adv_train_{student_stub}")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()

    monitor.start()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = rfgsm(model, xb, yb, eps=eps, alpha=alpha)
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)
                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                La_mean = torch.stack(adv_losses).mean() if len(adv_losses) > 0 else 0.0 * Lc
                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                L.backward()
                opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean += Lc.item() * bs
                tot_adv_mean += La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(
                f"[FGSM train {student_stub}] "
                f"epoch {ep}/{epochs} | clean={clean_loss_epoch:.4f} "
                f"| adv_mean={adv_mean_epoch:.4f} | ep={ep_mins:.2f}m | total={total_mins:.2f}m"
            )

            ep_ckpt = os.path.join(out_dir, f"{student_stub}{out_suffix}_epoch{ep:02d}.pth")
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    out = os.path.join(
        out_dir,
        os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth")
    )
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved FGSM-adv student:", out)

    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved FGSM train history to:", hist_path)

    summary_row = monitor.summary()
    summary_row.update({
        "student": student_stub,
        "attack_train_type": "fgsm",
        "epochs": epochs,
        "learning_rate": lr,
        "total_elapsed_minutes": (time.time() - start_time) / 60.0,
    })
    append_summary_row(summary_row, FGSM_TRAIN_RESOURCE_CSV)
    print("Updated FGSM student training resource summary:", FGSM_TRAIN_RESOURCE_CSV)

    return out


def adv_train_student_bim(student_path, out_dir, logs_dir,
                          out_suffix="_bim_adv",
                          epochs=5, lr=1e-3,
                          eps_list=None, iters=10, q=0.5,
                          random_start=False):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = BIM_TRAIN_EPS_LIST

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"bim_adv_train_{student_stub}")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()

    monitor.start()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = bim_attack(
                        model, xb, yb,
                        eps=eps,
                        alpha=eps / max(1, iters),
                        iters=iters,
                        random_start=random_start
                    )
                    model.train()
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)
                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                La_mean = torch.stack(adv_losses).mean() if len(adv_losses) > 0 else 0.0 * Lc
                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                L.backward()
                opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean += Lc.item() * bs
                tot_adv_mean += La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(
                f"[BIM train {student_stub}] "
                f"epoch {ep}/{epochs} | clean={clean_loss_epoch:.4f} "
                f"| adv_mean={adv_mean_epoch:.4f} | ep={ep_mins:.2f}m | total={total_mins:.2f}m"
            )

            ep_ckpt = os.path.join(out_dir, f"{student_stub}{out_suffix}_epoch{ep:02d}.pth")
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    out = os.path.join(
        out_dir,
        os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth")
    )
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved BIM-adv student:", out)

    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved BIM train history to:", hist_path)

    summary_row = monitor.summary()
    summary_row.update({
        "student": student_stub,
        "attack_train_type": "bim",
        "epochs": epochs,
        "learning_rate": lr,
        "iters": iters,
        "total_elapsed_minutes": (time.time() - start_time) / 60.0,
    })
    append_summary_row(summary_row, BIM_TRAIN_RESOURCE_CSV)
    print("Updated BIM student training resource summary:", BIM_TRAIN_RESOURCE_CSV)

    return out


def adv_train_student_pgd(student_path, out_dir, logs_dir,
                          out_suffix="_pgd_adv",
                          epochs=5, lr=1e-3,
                          eps_list=None, iters=10, q=0.5,
                          random_start=True):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = PGD_TRAIN_EPS_LIST

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"pgd_adv_train_{student_stub}")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()

    monitor.start()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = pgd_attack(
                        model, xb, yb,
                        eps=eps,
                        alpha=eps / max(1, iters),
                        iters=iters,
                        random_start=random_start
                    )
                    model.train()
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)
                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                La_mean = torch.stack(adv_losses).mean() if len(adv_losses) > 0 else 0.0 * Lc
                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                L.backward()
                opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean += Lc.item() * bs
                tot_adv_mean += La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(
                f"[PGD train {student_stub}] "
                f"epoch {ep}/{epochs} | clean={clean_loss_epoch:.4f} "
                f"| adv_mean={adv_mean_epoch:.4f} | ep={ep_mins:.2f}m | total={total_mins:.2f}m"
            )

            ep_ckpt = os.path.join(out_dir, f"{student_stub}{out_suffix}_epoch{ep:02d}.pth")
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    out = os.path.join(
        out_dir,
        os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth")
    )
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved PGD-adv student:", out)

    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved PGD train history to:", hist_path)

    summary_row = monitor.summary()
    summary_row.update({
        "student": student_stub,
        "attack_train_type": "pgd",
        "epochs": epochs,
        "learning_rate": lr,
        "iters": iters,
        "total_elapsed_minutes": (time.time() - start_time) / 60.0,
    })
    append_summary_row(summary_row, PGD_TRAIN_RESOURCE_CSV)
    print("Updated PGD student training resource summary:", PGD_TRAIN_RESOURCE_CSV)

    return out

In [ ]:
# attacks

def rfgsm(model, X, Y, eps=0.1, alpha=0.02):
    was_training = model.training
    model.eval()

    X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0, 1).detach()
    X_adv.requires_grad_(True)

    loss = mse(model(X_adv), Y)
    loss.backward()

    X_adv = (X_adv + alpha * X_adv.grad.sign()).clamp(0, 1)

    if was_training:
        model.train()

    return X_adv.detach()


def bim_attack(model, X, Y, eps=0.15, alpha=None, iters=10, random_start=False):
    if alpha is None:
        alpha = eps / max(1, iters)

    was_training = model.training
    model.eval()

    if random_start:
        X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0, 1).detach()
    else:
        X_adv = X.clone().detach()

    X_adv.requires_grad_(True)

    for _ in range(iters):
        preds = model(X_adv)
        loss = mse(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        grad_sign = X_adv.grad.sign()
        X_adv = (X_adv + alpha * grad_sign).detach()

        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0, 1).detach()
        X_adv.requires_grad_(True)

    if was_training:
        model.train()

    return X_adv.detach()


def pgd_attack(model, X, Y, eps=0.15, alpha=None, iters=10, random_start=True):
    if alpha is None:
        alpha = eps / max(1, iters)

    was_training = model.training
    model.eval()

    if random_start:
        X_adv = (X + torch.empty_like(X).uniform_(-eps, eps)).clamp(0, 1).detach()
    else:
        X_adv = X.clone().detach()

    X_adv.requires_grad_(True)

    for _ in range(iters):
        preds = model(X_adv)
        loss = mse(preds, Y)

        if X_adv.grad is not None:
            X_adv.grad.zero_()

        loss.backward()
        grad_sign = X_adv.grad.sign()
        X_adv = (X_adv + alpha * grad_sign).detach()

        delta = torch.clamp(X_adv - X, -eps, eps)
        X_adv = (X + delta).clamp(0, 1).detach()
        X_adv.requires_grad_(True)

    if was_training:
        model.train()

    return X_adv.detach()

In [ ]:
# adversarial training

print("\nTraining FGSM adv students")
fgsm_adv_paths = []
for sp in base_student_paths:
    out = adv_train_student_fgsm(
        sp,
        out_dir=FGSM_ADV_DIR,
        logs_dir=FGSM_LOGS_DIR,
        out_suffix="_fgsm_adv",
        epochs=FGSM_EPOCHS,
        lr=LR_STUDENT,
        eps_list=FGSM_TRAIN_EPS_LIST,
        alpha=0.02,
    )
    fgsm_adv_paths.append(out)

print("\nTraining BIM adv students")
bim_adv_paths = []
for sp in base_student_paths:
    out = adv_train_student_bim(
        sp,
        out_dir=BIM_ADV_DIR,
        logs_dir=BIM_LOGS_DIR,
        out_suffix="_bim_adv",
        epochs=BIM_EPOCHS,
        lr=LR_STUDENT,
        eps_list=BIM_TRAIN_EPS_LIST,
        iters=BIM_ITERS,
        random_start=False,
    )
    bim_adv_paths.append(out)

print("\nTraining PGD adv students")
pgd_adv_paths = []
for sp in base_student_paths:
    out = adv_train_student_pgd(
        sp,
        out_dir=PGD_ADV_DIR,
        logs_dir=PGD_LOGS_DIR,
        out_suffix="_pgd_adv",
        epochs=PGD_EPOCHS,
        lr=LR_STUDENT,
        eps_list=PGD_TRAIN_EPS_LIST,
        iters=PGD_ITERS,
        random_start=True,
    )
    pgd_adv_paths.append(out)


Training FGSM adv students


/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[FGSM train student_base_01] epoch 1/5 | clean=0.0329 | adv_mean=0.0925 | ep=0.35m | total=0.35m
[FGSM train student_base_01] epoch 2/5 | clean=0.0291 | adv_mean=0.0888 | ep=0.35m | total=0.70m
[FGSM train student_base_01] epoch 3/5 | clean=0.0268 | adv_mean=0.0877 | ep=0.35m | total=1.05m
[FGSM train student_base_01] epoch 4/5 | clean=0.0261 | adv_mean=0.0867 | ep=0.35m | total=1.40m
[FGSM train student_base_01] epoch 5/5 | clean=0.0246 | adv_mean=0.0858 | ep=0.35m | total=1.74m
Saved FGSM-adv student: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/advStudents/student_base_01_fgsm_adv.pth
Saved FGSM train history to: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/results/train_logs/student_base_01_fgsm_adv_train_history.csv
Updated FGSM student training resource summary: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/fgsm_student_train_resource_summary.csv


/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[FGSM train student_base_02] epoch 1/5 | clean=0.0325 | adv_mean=0.0926 | ep=0.36m | total=0.36m
[FGSM train student_base_02] epoch 2/5 | clean=0.0286 | adv_mean=0.0889 | ep=0.35m | total=0.71m
[FGSM train student_base_02] epoch 3/5 | clean=0.0264 | adv_mean=0.0875 | ep=0.35m | total=1.05m
[FGSM train student_base_02] epoch 4/5 | clean=0.0254 | adv_mean=0.0864 | ep=0.35m | total=1.41m
[FGSM train student_base_02] epoch 5/5 | clean=0.0249 | adv_mean=0.0859 | ep=0.35m | total=1.76m
Saved FGSM-adv student: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/advStudents/student_base_02_fgsm_adv.pth
Saved FGSM train history to: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/results/train_logs/student_base_02_fgsm_adv_train_history.csv
Updated FGSM student training resource summary: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/fgsm_student_train_resource_summary.csv


...

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[PGD train student_base_03] epoch 1/5 | clean=0.1261 | adv_mean=0.2087 | ep=1.61m | total=1.61m
[PGD train student_base_03] epoch 2/5 | clean=0.1044 | adv_mean=0.2026 | ep=1.65m | total=3.26m
[PGD train student_base_03] epoch 3/5 | clean=0.0846 | adv_mean=0.1913 | ep=1.62m | total=4.88m
[PGD train student_base_03] epoch 4/5 | clean=0.0617 | adv_mean=0.1748 | ep=1.62m | total=6.50m
[PGD train student_base_03] epoch 5/5 | clean=0.0527 | adv_mean=0.1683 | ep=1.61m | total=8.11m
Saved PGD-adv student: /content/drive/MyDrive/298/Jena/proj_v3/pgd/advStudents/student_base_03_pgd_adv.pth
Saved PGD train history to: /content/drive/MyDrive/298/Jena/proj_v3/pgd/results/train_logs/student_base_03_pgd_adv_train_history.csv
Updated PGD student training resource summary: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/pgd_student_train_resource_summary.csv


/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[PGD train student_base_04] epoch 1/5 | clean=0.1314 | adv_mean=0.2089 | ep=1.60m | total=1.60m
[PGD train student_base_04] epoch 2/5 | clean=0.1038 | adv_mean=0.2014 | ep=1.60m | total=3.21m
[PGD train student_base_04] epoch 3/5 | clean=0.0811 | adv_mean=0.1807 | ep=1.61m | total=4.82m
[PGD train student_base_04] epoch 4/5 | clean=0.0565 | adv_mean=0.1590 | ep=1.61m | total=6.42m
[PGD train student_base_04] epoch 5/5 | clean=0.0599 | adv_mean=0.1565 | ep=1.61m | total=8.04m
Saved PGD-adv student: /content/drive/MyDrive/298/Jena/proj_v3/pgd/advStudents/student_base_04_pgd_adv.pth
Saved PGD train history to: /content/drive/MyDrive/298/Jena/proj_v3/pgd/results/train_logs/student_base_04_pgd_adv_train_history.csv
Updated PGD student training resource summary: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/pgd_student_train_resource_summary.csv


/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
# eval helpers

def eval_clean_rmse(model, loader):
    model.eval()
    ys, ps = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            p = model(xb)
            ys.append(yb.cpu().numpy())
            ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))


def eval_ensemble_rmse(models, loader):
    for m in models:
        m.eval()

    ys, ps = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            preds = []
            for m in models:
                preds.append(m(xb))

            preds = torch.stack(preds, dim=0).mean(dim=0)
            ys.append(yb.cpu().numpy())
            ps.append(preds.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))


def eval_pair_attack(defender, attacker, loader, attack_fn, atk_kwargs):
    defender.eval()
    attacker.eval()

    ys, ps = [], []

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        xa = attack_fn(attacker, xb, yb, **atk_kwargs)

        with torch.no_grad():
            p = defender(xa)

        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))


def eval_random_switch_attack(models, loader, attack_fn, atk_kwargs_func):
    for m in models:
        m.eval()

    ys, ps = [], []

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        atk_model = random.choice(models)
        def_model = random.choice(models)

        atk_kwargs = atk_kwargs_func()
        xa = attack_fn(atk_model, xb, yb, **atk_kwargs)

        with torch.no_grad():
            p = def_model(xa)

        ys.append(yb.cpu().numpy())
        ps.append(p.cpu().numpy())

    ys = np.vstack(ys)
    ps = np.vstack(ps)
    return float(np.sqrt(mean_squared_error(ys, ps)))

In [ ]:
# diversity metrics

def flatten_params(model):
    return torch.cat([p.detach().cpu().view(-1) for p in model.parameters()])


def compute_weight_diversity(students, names, out_csv):
    vecs = [flatten_params(m) for m in students]
    k = len(vecs)

    rows = []
    for i, j in itertools.combinations(range(k), 2):
        d = torch.norm(vecs[i] - vecs[j], p=2).item()
        rows.append({
            "student_i": names[i],
            "student_j": names[j],
            "l2_distance": d,
        })

    df_w = pd.DataFrame(rows)
    df_w.to_csv(out_csv, index=False)
    print("Saved weight diversity to:", out_csv)
    return df_w


@torch.no_grad()
def compute_prediction_diversity(students, loader, out_csv):
    for m in students:
        m.eval()

    k = len(students)
    sum_pred = None
    sum_pred2 = None
    count = 0

    for xb, yb in loader:
        xb = xb.to(device)

        preds_stu = []
        for m in students:
            p = m(xb).cpu().numpy().reshape(-1)
            preds_stu.append(p)

        preds_stu = np.stack(preds_stu, axis=0)
        preds_batch_mean = preds_stu.mean(axis=1)

        if sum_pred is None:
            sum_pred = np.zeros_like(preds_batch_mean, dtype=np.float64)
            sum_pred2 = np.zeros_like(preds_batch_mean, dtype=np.float64)

        sum_pred += preds_batch_mean
        sum_pred2 += preds_batch_mean ** 2
        count += 1

    if count == 0:
        raise RuntimeError("compute_prediction_diversity: loader produced 0 batches")

    mean_pred = sum_pred / count
    mean_pred2 = sum_pred2 / count
    var_pred = np.maximum(mean_pred2 - mean_pred ** 2, 0.0)

    df_p = pd.DataFrame({
        "student_idx": np.arange(k),
        "mean_pred": mean_pred,
        "var_pred": var_pred,
    })
    df_p.to_csv(out_csv, index=False)
    print("Saved prediction diversity to:", out_csv)
    return df_p

In [ ]:
# load students once

def load_model_list(path_list):
    models = []
    for p in path_list:
        m = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
        m.load_state_dict(torch.load(p, map_location=device))
        m.eval()
        models.append(m)
    return models


base_eval = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
base_eval.load_state_dict(torch.load(base_best_ckpt, map_location=device))
base_eval.eval()

fgsm_adv_paths = sorted([
    os.path.join(FGSM_ADV_DIR, f)
    for f in os.listdir(FGSM_ADV_DIR)
    if f.endswith("_fgsm_adv.pth")
])

bim_adv_paths = sorted([
    os.path.join(BIM_ADV_DIR, f)
    for f in os.listdir(BIM_ADV_DIR)
    if f.endswith("_bim_adv.pth")
])

pgd_adv_paths = sorted([
    os.path.join(PGD_ADV_DIR, f)
    for f in os.listdir(PGD_ADV_DIR)
    if f.endswith("_pgd_adv.pth")
])

print("FGSM adv students found:", len(fgsm_adv_paths))
print("BIM adv students found :", len(bim_adv_paths))
print("PGD adv students found :", len(pgd_adv_paths))

fgsm_students = load_model_list(fgsm_adv_paths)
bim_students  = load_model_list(bim_adv_paths)
pgd_students  = load_model_list(pgd_adv_paths)

fgsm_student_names = [os.path.basename(p).replace(".pth", "") for p in fgsm_adv_paths]
bim_student_names  = [os.path.basename(p).replace(".pth", "") for p in bim_adv_paths]
pgd_student_names  = [os.path.basename(p).replace(".pth", "") for p in pgd_adv_paths]

print("\nLoaded adv student pools:")
print(" FGSM:", len(fgsm_students))
print(" BIM :", len(bim_students))
print(" PGD :", len(pgd_students))

FGSM adv students found: 4
BIM adv students found : 4
PGD adv students found : 4

Loaded adv student pools:
 FGSM: 4
 BIM : 4
 PGD : 4


/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [ ]:
# transferability matrix with computational metrics

def compute_transfer_matrix(
    students,
    names,
    loader,
    attack_fn,
    eps_list,
    extra_kwargs_func,
    out_csv,
    attack_label="rfgsm",
    trace_csv=None,
    summary_csv=None,
):
    rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_label}_transfer_matrix")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()

    monitor.start()

    try:
        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)
            print(f"\n[{attack_label.upper()}] Transfer matrix for eps={eps}...")

            for i, atk_name in enumerate(names):
                for j, def_name in enumerate(names):
                    atk_model = students[i]
                    def_model = students[j]

                    rmse_ij = eval_pair_attack(
                        defender=def_model,
                        attacker=atk_model,
                        loader=loader,
                        attack_fn=attack_fn,
                        atk_kwargs=atk_kwargs,
                    )

                    rows.append({
                        "attack": attack_label,
                        "epsilon": eps,
                        "attacker": atk_name,
                        "defender": def_name,
                        "rmse_adv": rmse_ij,
                    })

                    print(
                        f"  atk={atk_name} - def={def_name} | "
                        f"eps={eps:.2f} | RMSE={rmse_ij:.5f}"
                    )
    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {attack_label} transfer matrix to:", out_csv)

    if trace_csv is not None and summary_csv is not None:
        save_monitor_outputs(
            monitor,
            trace_csv=trace_csv,
            summary_csv=summary_csv,
            extra_summary={
                "stage": f"{attack_label}_transfer_matrix",
                "total_elapsed_minutes": (time.time() - start_time) / 60.0,
            }
        )

    return df

In [ ]:
# single-pass morphence eval with computational metrics
# replaces the old 30-run evaluation

def run_morphence_eval_single_pass(
    attack_name,
    attack_fn,
    eps_list,
    students,
    results_csv,
    stats_csv,
    extra_kwargs_func,
    trace_csv,
    summary_csv,
):
    all_rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_name}_single_pass_eval")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_all = time.time()

    monitor.start()

    try:
        base_clean = eval_clean_rmse(base_eval, test_dl)
        morph_clean = eval_ensemble_rmse(students, test_dl)

        all_rows.append(["clean", None, "base", base_clean, 1, SEED])
        all_rows.append(["clean", None, "morph_ensemble", morph_clean, 1, SEED])

        print(
            f"[{attack_name.upper()}] clean | "
            f"base={base_clean:.5f} | morph_ensemble={morph_clean:.5f}"
        )

        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)

            base_rmse = eval_pair_attack(
                defender=base_eval,
                attacker=base_eval,
                loader=test_dl,
                attack_fn=attack_fn,
                atk_kwargs=atk_kwargs,
            )

            def atk_kwargs_wrapper():
                return atk_kwargs

            morph_rmse = eval_random_switch_attack(
                models=students,
                loader=test_dl,
                attack_fn=attack_fn,
                atk_kwargs_func=atk_kwargs_wrapper,
            )

            all_rows.append([attack_name, eps, "base", base_rmse, 1, SEED])
            all_rows.append([attack_name, eps, "morph_rand", morph_rmse, 1, SEED])

            print(
                f"[{attack_name.upper()}] eps={eps:.2f} | "
                f"base={base_rmse:.5f} | morph_rand={morph_rmse:.5f}"
            )

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    df_runs = pd.DataFrame(
        all_rows,
        columns=["attack", "epsilon", "model", "RMSE", "run", "seed"]
    )
    df_runs.to_csv(results_csv, index=False)

    stats = (
        df_runs
        .groupby(["attack", "epsilon", "model"])["RMSE"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
        .sort_values(["attack", "epsilon", "model"])
    )
    stats.to_csv(stats_csv, index=False)

    save_monitor_outputs(
        monitor,
        trace_csv=trace_csv,
        summary_csv=summary_csv,
        extra_summary={
            "stage": f"{attack_name}_single_pass_eval",
            "total_elapsed_minutes": (time.time() - start_all) / 60.0,
        }
    )

    print(f"\nSaved {attack_name} runs to:", results_csv)
    print(f"Saved {attack_name} stats to:", stats_csv)

    return df_runs, stats

In [ ]:
# attack kwarg helpers

def fgsm_kwargs(eps):
    return {
        "eps": eps,
        "alpha": 0.02,
    }

def bim_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / BIM_ITERS,
        "iters": BIM_ITERS,
        "random_start": False,
    }

def pgd_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / PGD_ITERS,
        "iters": PGD_ITERS,
        "random_start": True,
    }

In [ ]:
# FGSM diversity + transferability + single-pass eval

print("\nFGSM diversity + transferability + single-pass eval")

_ = compute_weight_diversity(fgsm_students, fgsm_student_names, FGSM_WEIGHT_DIVERSITY_CSV)
_ = compute_prediction_diversity(fgsm_students, test_dl, FGSM_PRED_DIVERSITY_CSV)

_ = compute_transfer_matrix(
    students=fgsm_students,
    names=fgsm_student_names,
    loader=test_dl,
    attack_fn=rfgsm,
    eps_list=EPS_LIST,
    extra_kwargs_func=fgsm_kwargs,
    out_csv=FGSM_XFER_MATRIX_CSV,
    attack_label="rfgsm",
    trace_csv=FGSM_XFER_TRACE_CSV,
    summary_csv=FGSM_XFER_SUMMARY_CSV,
)

df_fgsm_runs, fgsm_stats = run_morphence_eval_single_pass(
    attack_name="rfgsm",
    attack_fn=rfgsm,
    eps_list=EPS_LIST,
    students=fgsm_students,
    results_csv=FGSM_RUNS_CSV,
    stats_csv=FGSM_STATS_CSV,
    extra_kwargs_func=fgsm_kwargs,
    trace_csv=FGSM_EVAL_TRACE_CSV,
    summary_csv=FGSM_EVAL_SUMMARY_CSV,
)


FGSM diversity + transferability + single-pass eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/results/fgsm_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/results/fgsm_prediction_diversity.csv

[RFGSM] Transfer matrix for eps=0.1...
  atk=student_base_01_fgsm_adv - def=student_base_01_fgsm_adv | eps=0.10 | RMSE=0.04717
  atk=student_base_01_fgsm_adv - def=student_base_02_fgsm_adv | eps=0.10 | RMSE=0.04666
  atk=student_base_01_fgsm_adv - def=student_base_03_fgsm_adv | eps=0.10 | RMSE=0.04522
  atk=student_base_01_fgsm_adv - def=student_base_04_fgsm_adv | eps=0.10 | RMSE=0.04723
  atk=student_base_02_fgsm_adv - def=student_base_01_fgsm_adv | eps=0.10 | RMSE=0.04549
  atk=student_base_02_fgsm_adv - def=student_base_02_fgsm_adv | eps=0.10 | RMSE=0.04759
  atk=student_base_02_fgsm_adv - def=student_base_03_fgsm_adv | eps=0.10 | RMSE=0.04503
  atk=student_base_02_fgsm_adv - def=student_base_04_fgsm_adv | eps=0.1

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[RFGSM] clean | base=0.01484 | morph_ensemble=0.01615
[RFGSM] eps=0.10 | base=0.07300 | morph_rand=0.04575
[RFGSM] eps=0.20 | base=0.09804 | morph_rand=0.05257
[RFGSM] eps=0.30 | base=0.12001 | morph_rand=0.05833
[RFGSM] eps=0.50 | base=0.14815 | morph_rand=0.07251
Saved resource trace to: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/fgsm_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/fgsm_eval_resource_summary.csv
Resource summary: {'n_samples': 5, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1834.390625, 'max_gpu_mem_mb': 977.0, 'avg_gpu_util_percent': 30.8, 'gpu_active_percent_of_samples': 80.0, 'est_gpu_energy_j': 1132.4890284597873, 'stage': 'rfgsm_single_pass_eval', 'total_elapsed_minutes': 0.18496453762054443}

Saved rfgsm runs to: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/results/fgsm_morphence_runs.csv
Saved rfgsm stats to: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/results/fgsm_morphence_

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
# BIM diversity + transferability + single-pass eval

print("\nBIM diversity + transferability + single-pass eval")

_ = compute_weight_diversity(bim_students, bim_student_names, BIM_WEIGHT_DIVERSITY_CSV)
_ = compute_prediction_diversity(bim_students, test_dl, BIM_PRED_DIVERSITY_CSV)

_ = compute_transfer_matrix(
    students=bim_students,
    names=bim_student_names,
    loader=test_dl,
    attack_fn=bim_attack,
    eps_list=EPS_LIST,
    extra_kwargs_func=bim_kwargs,
    out_csv=BIM_XFER_MATRIX_CSV,
    attack_label="bim",
    trace_csv=BIM_XFER_TRACE_CSV,
    summary_csv=BIM_XFER_SUMMARY_CSV,
)

df_bim_runs, bim_stats = run_morphence_eval_single_pass(
    attack_name="bim",
    attack_fn=bim_attack,
    eps_list=EPS_LIST,
    students=bim_students,
    results_csv=BIM_RUNS_CSV,
    stats_csv=BIM_STATS_CSV,
    extra_kwargs_func=bim_kwargs,
    trace_csv=BIM_EVAL_TRACE_CSV,
    summary_csv=BIM_EVAL_SUMMARY_CSV,
)


BIM diversity + transferability + single-pass eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v3/bim/results/bim_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v3/bim/results/bim_prediction_diversity.csv

[BIM] Transfer matrix for eps=0.1...
  atk=student_base_01_bim_adv - def=student_base_01_bim_adv | eps=0.10 | RMSE=0.14448
  atk=student_base_01_bim_adv - def=student_base_02_bim_adv | eps=0.10 | RMSE=0.14004
  atk=student_base_01_bim_adv - def=student_base_03_bim_adv | eps=0.10 | RMSE=0.14009
  atk=student_base_01_bim_adv - def=student_base_04_bim_adv | eps=0.10 | RMSE=0.14009
  atk=student_base_02_bim_adv - def=student_base_01_bim_adv | eps=0.10 | RMSE=0.14446
  atk=student_base_02_bim_adv - def=student_base_02_bim_adv | eps=0.10 | RMSE=0.14004
  atk=student_base_02_bim_adv - def=student_base_03_bim_adv | eps=0.10 | RMSE=0.14010
  atk=student_base_02_bim_adv - def=student_base_04_bim_adv | eps=0.10 | RMSE=0.14009
  atk=

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[BIM] clean | base=0.01484 | morph_ensemble=0.14103
[BIM] eps=0.10 | base=0.14495 | morph_rand=0.14114
[BIM] eps=0.20 | base=0.24547 | morph_rand=0.14098
[BIM] eps=0.30 | base=0.33308 | morph_rand=0.14065
[BIM] eps=0.50 | base=0.45695 | morph_rand=0.14114
Saved resource trace to: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/bim_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/bim_eval_resource_summary.csv
Resource summary: {'n_samples': 26, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1835.15625, 'max_gpu_mem_mb': 977.0, 'avg_gpu_util_percent': 44.34615384615385, 'gpu_active_percent_of_samples': 96.15384615384616, 'est_gpu_energy_j': 8161.414007314443, 'stage': 'bim_single_pass_eval', 'total_elapsed_minutes': 0.8902009844779968}

Saved bim runs to: /content/drive/MyDrive/298/Jena/proj_v3/bim/results/bim_morphence_runs.csv
Saved bim stats to: /content/drive/MyDrive/298/Jena/proj_v3/bim/results/bim_morphenc

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
# PGD diversity + transferability + single-pass eval

print("\nPGD diversity + transferability + single-pass eval")

_ = compute_weight_diversity(pgd_students, pgd_student_names, PGD_WEIGHT_DIVERSITY_CSV)
_ = compute_prediction_diversity(pgd_students, test_dl, PGD_PRED_DIVERSITY_CSV)

_ = compute_transfer_matrix(
    students=pgd_students,
    names=pgd_student_names,
    loader=test_dl,
    attack_fn=pgd_attack,
    eps_list=EPS_LIST,
    extra_kwargs_func=pgd_kwargs,
    out_csv=PGD_XFER_MATRIX_CSV,
    attack_label="pgd",
    trace_csv=PGD_XFER_TRACE_CSV,
    summary_csv=PGD_XFER_SUMMARY_CSV,
)

df_pgd_runs, pgd_stats = run_morphence_eval_single_pass(
    attack_name="pgd",
    attack_fn=pgd_attack,
    eps_list=EPS_LIST,
    students=pgd_students,
    results_csv=PGD_RUNS_CSV,
    stats_csv=PGD_STATS_CSV,
    extra_kwargs_func=pgd_kwargs,
    trace_csv=PGD_EVAL_TRACE_CSV,
    summary_csv=PGD_EVAL_SUMMARY_CSV,
)


PGD diversity + transferability + single-pass eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v3/pgd/results/pgd_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v3/pgd/results/pgd_prediction_diversity.csv

[PGD] Transfer matrix for eps=0.1...
  atk=student_base_01_pgd_adv - def=student_base_01_pgd_adv | eps=0.10 | RMSE=0.12606
  atk=student_base_01_pgd_adv - def=student_base_02_pgd_adv | eps=0.10 | RMSE=0.09066
  atk=student_base_01_pgd_adv - def=student_base_03_pgd_adv | eps=0.10 | RMSE=0.09335
  atk=student_base_01_pgd_adv - def=student_base_04_pgd_adv | eps=0.10 | RMSE=0.10172
  atk=student_base_02_pgd_adv - def=student_base_01_pgd_adv | eps=0.10 | RMSE=0.08755
  atk=student_base_02_pgd_adv - def=student_base_02_pgd_adv | eps=0.10 | RMSE=0.11996
  atk=student_base_02_pgd_adv - def=student_base_03_pgd_adv | eps=0.10 | RMSE=0.08982
  atk=student_base_02_pgd_adv - def=student_base_04_pgd_adv | eps=0.10 | RMSE=0.11805
  atk=

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[PGD] clean | base=0.01484 | morph_ensemble=0.02530
[PGD] eps=0.10 | base=0.12716 | morph_rand=0.10937
[PGD] eps=0.20 | base=0.21736 | morph_rand=0.14341
[PGD] eps=0.30 | base=0.29937 | morph_rand=0.15412
[PGD] eps=0.50 | base=0.42782 | morph_rand=0.17134
Saved resource trace to: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/pgd_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/pgd_eval_resource_summary.csv
Resource summary: {'n_samples': 26, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1835.2421875, 'max_gpu_mem_mb': 977.0, 'avg_gpu_util_percent': 44.80769230769231, 'gpu_active_percent_of_samples': 96.15384615384616, 'est_gpu_energy_j': 8140.32528268814, 'stage': 'pgd_single_pass_eval', 'total_elapsed_minutes': 0.8901840170224508}

Saved pgd runs to: /content/drive/MyDrive/298/Jena/proj_v3/pgd/results/pgd_morphence_runs.csv
Saved pgd stats to: /content/drive/MyDrive/298/Jena/proj_v3/pgd/results/pgd_morphen

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
print("\n================ ORIGINAL RESULT OUTPUTS ================\n")

print("FGSM runs:", FGSM_RUNS_CSV)
print("BIM  runs:", BIM_RUNS_CSV)
print("PGD  runs:", PGD_RUNS_CSV)

print("FGSM weight diversity:", FGSM_WEIGHT_DIVERSITY_CSV)
print("BIM  weight diversity:", BIM_WEIGHT_DIVERSITY_CSV)
print("PGD  weight diversity:", PGD_WEIGHT_DIVERSITY_CSV)

print("FGSM prediction diversity:", FGSM_PRED_DIVERSITY_CSV)
print("BIM  prediction diversity:", BIM_PRED_DIVERSITY_CSV)
print("PGD  prediction diversity:", PGD_PRED_DIVERSITY_CSV)

print("FGSM transfer matrix:", FGSM_XFER_MATRIX_CSV)
print("BIM  transfer matrix:", BIM_XFER_MATRIX_CSV)
print("PGD  transfer matrix:", PGD_XFER_MATRIX_CSV)

print("\n================ COMPUTATIONAL METRICS OUTPUTS ================\n")

print("Base training trace       :", BASE_RESOURCE_TRACE_CSV)
print("Base training summary     :", BASE_RESOURCE_SUMMARY_CSV)
print("Base model profile        :", BASE_PROFILE_CSV)

print("FGSM student train summary:", FGSM_TRAIN_RESOURCE_CSV)
print("BIM student train summary :", BIM_TRAIN_RESOURCE_CSV)
print("PGD student train summary :", PGD_TRAIN_RESOURCE_CSV)

print("FGSM transfer trace       :", FGSM_XFER_TRACE_CSV)
print("FGSM transfer summary     :", FGSM_XFER_SUMMARY_CSV)
print("BIM transfer trace        :", BIM_XFER_TRACE_CSV)
print("BIM transfer summary      :", BIM_XFER_SUMMARY_CSV)
print("PGD transfer trace        :", PGD_XFER_TRACE_CSV)
print("PGD transfer summary      :", PGD_XFER_SUMMARY_CSV)

print("FGSM eval trace           :", FGSM_EVAL_TRACE_CSV)
print("FGSM eval summary         :", FGSM_EVAL_SUMMARY_CSV)
print("BIM eval trace            :", BIM_EVAL_TRACE_CSV)
print("BIM eval summary          :", BIM_EVAL_SUMMARY_CSV)
print("PGD eval trace            :", PGD_EVAL_TRACE_CSV)
print("PGD eval summary          :", PGD_EVAL_SUMMARY_CSV)


================ ORIGINAL RESULT OUTPUTS ================

FGSM runs: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/results/fgsm_morphence_runs.csv
BIM  runs: /content/drive/MyDrive/298/Jena/proj_v3/bim/results/bim_morphence_runs.csv
PGD  runs: /content/drive/MyDrive/298/Jena/proj_v3/pgd/results/pgd_morphence_runs.csv
FGSM weight diversity: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/results/fgsm_weight_diversity.csv
BIM  weight diversity: /content/drive/MyDrive/298/Jena/proj_v3/bim/results/bim_weight_diversity.csv
PGD  weight diversity: /content/drive/MyDrive/298/Jena/proj_v3/pgd/results/pgd_weight_diversity.csv
FGSM prediction diversity: /content/drive/MyDrive/298/Jena/proj_v3/fgsm/results/fgsm_prediction_diversity.csv
BIM  prediction diversity: /content/drive/MyDrive/298/Jena/proj_v3/bim/results/bim_prediction_diversity.csv
PGD  prediction diversity: /content/drive/MyDrive/298/Jena/proj_v3/pgd/results/pgd_prediction_diversity.csv
FGSM transfer matrix: /content/drive/MyDrive/298/Jen

In [ ]:
# =========================================================
# Jena novel-student continuation for proj_v3
# =========================================================

BASE_PROJ_DIR = "/content/drive/MyDrive/298/Jena/proj_v3"

JENANOV_DIR      = os.path.join(BASE_PROJ_DIR, "jenaNov")
STUDENTS_NOV_DIR = os.path.join(JENANOV_DIR, "studentsNov")
JENANOV_LOGS_DIR = os.path.join(JENANOV_DIR, "logs")

# fgsm nov
FGSM_NOV_DIR              = os.path.join(JENANOV_DIR, "fgsm")
FGSM_NOV_STUDENTS_ADV_DIR = os.path.join(FGSM_NOV_DIR, "studentsNovAdv")
FGSM_NOV_RESULTS_DIR      = os.path.join(FGSM_NOV_DIR, "results")
FGSM_NOV_LOGS_DIR         = os.path.join(FGSM_NOV_RESULTS_DIR, "train_logs")

NOV_FGSM_RUNS_CSV         = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_morphence_runs_single.csv")
NOV_FGSM_STATS_CSV        = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_morphence_stats_single.csv")
NOV_FGSM_WEIGHT_DIVERSITY = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_weight_diversity.csv")
NOV_FGSM_PRED_DIVERSITY   = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_prediction_diversity.csv")
NOV_FGSM_XFER_MATRIX_CSV  = os.path.join(FGSM_NOV_RESULTS_DIR, "nov_fgsm_transfer_matrix.csv")

# bim nov
BIM_NOV_DIR              = os.path.join(JENANOV_DIR, "bim")
BIM_NOV_STUDENTS_ADV_DIR = os.path.join(BIM_NOV_DIR, "studentsNovAdv")
BIM_NOV_RESULTS_DIR      = os.path.join(BIM_NOV_DIR, "results")
BIM_NOV_LOGS_DIR         = os.path.join(BIM_NOV_RESULTS_DIR, "train_logs")
BIM_NOV_CRASH_DIR        = os.path.join(BIM_NOV_DIR, "crash_dump")

NOV_BIM_RUNS_CSV         = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_morphence_runs_single.csv")
NOV_BIM_STATS_CSV        = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_morphence_stats_single.csv")
NOV_BIM_WEIGHT_DIVERSITY = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_weight_diversity.csv")
NOV_BIM_PRED_DIVERSITY   = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_prediction_diversity.csv")
NOV_BIM_XFER_MATRIX_CSV  = os.path.join(BIM_NOV_RESULTS_DIR, "nov_bim_transfer_matrix.csv")

# pgd nov
PGD_NOV_DIR              = os.path.join(JENANOV_DIR, "pgd")
PGD_NOV_STUDENTS_ADV_DIR = os.path.join(PGD_NOV_DIR, "studentsNovAdv")
PGD_NOV_RESULTS_DIR      = os.path.join(PGD_NOV_DIR, "results")
PGD_NOV_LOGS_DIR         = os.path.join(PGD_NOV_RESULTS_DIR, "train_logs")
PGD_NOV_CRASH_DIR        = os.path.join(PGD_NOV_DIR, "crash_dump")

NOV_PGD_RUNS_CSV         = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_morphence_runs_single.csv")
NOV_PGD_STATS_CSV        = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_morphence_stats_single.csv")
NOV_PGD_WEIGHT_DIVERSITY = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_weight_diversity.csv")
NOV_PGD_PRED_DIVERSITY   = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_prediction_diversity.csv")
NOV_PGD_XFER_MATRIX_CSV  = os.path.join(PGD_NOV_RESULTS_DIR, "nov_pgd_transfer_matrix.csv")

for d in [
    JENANOV_DIR, STUDENTS_NOV_DIR, JENANOV_LOGS_DIR,
    FGSM_NOV_DIR, FGSM_NOV_STUDENTS_ADV_DIR, FGSM_NOV_RESULTS_DIR, FGSM_NOV_LOGS_DIR,
    BIM_NOV_DIR, BIM_NOV_STUDENTS_ADV_DIR, BIM_NOV_RESULTS_DIR, BIM_NOV_LOGS_DIR, BIM_NOV_CRASH_DIR,
    PGD_NOV_DIR, PGD_NOV_STUDENTS_ADV_DIR, PGD_NOV_RESULTS_DIR, PGD_NOV_LOGS_DIR, PGD_NOV_CRASH_DIR,
]:
    os.makedirs(d, exist_ok=True)

# computational metrics for novel half
JENANOV_COMP_METRICS_DIR = os.path.join(JENANOV_DIR, "computational_metrics")
os.makedirs(JENANOV_COMP_METRICS_DIR, exist_ok=True)

MODEL_SIZES_NOV_CSV = os.path.join(JENANOV_DIR, "jenaNov_model_sizes.csv")

FGSM_NOV_TRAIN_RESOURCE_CSV = os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_student_train_resource_summary.csv")
BIM_NOV_TRAIN_RESOURCE_CSV  = os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_student_train_resource_summary.csv")
PGD_NOV_TRAIN_RESOURCE_CSV  = os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_student_train_resource_summary.csv")

FGSM_NOV_XFER_TRACE_CSV   = os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_transfer_resource_trace.csv")
FGSM_NOV_XFER_SUMMARY_CSV = os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_transfer_resource_summary.csv")
BIM_NOV_XFER_TRACE_CSV    = os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_transfer_resource_trace.csv")
BIM_NOV_XFER_SUMMARY_CSV  = os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_transfer_resource_summary.csv")
PGD_NOV_XFER_TRACE_CSV    = os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_transfer_resource_trace.csv")
PGD_NOV_XFER_SUMMARY_CSV  = os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_transfer_resource_summary.csv")

FGSM_NOV_EVAL_TRACE_CSV   = os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_eval_resource_trace.csv")
FGSM_NOV_EVAL_SUMMARY_CSV = os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_eval_resource_summary.csv")
BIM_NOV_EVAL_TRACE_CSV    = os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_eval_resource_trace.csv")
BIM_NOV_EVAL_SUMMARY_CSV  = os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_eval_resource_summary.csv")
PGD_NOV_EVAL_TRACE_CSV    = os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_eval_resource_trace.csv")
PGD_NOV_EVAL_SUMMARY_CSV  = os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_eval_resource_summary.csv")

print("BASE_PROJ_DIR :", BASE_PROJ_DIR)
print("JENANOV_DIR   :", JENANOV_DIR)
print("JENANOV COMP  :", JENANOV_COMP_METRICS_DIR)

BASE_PROJ_DIR : /content/drive/MyDrive/298/Jena/proj_v3
JENANOV_DIR   : /content/drive/MyDrive/298/Jena/proj_v3/jenaNov
JENANOV COMP  : /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics


In [ ]:
# load base model from proj_v3 baseModel

base_best_ckpt = os.path.join(BASE_PROJ_DIR, "baseModel", "jena_base_best.pth")
assert os.path.exists(base_best_ckpt), f"Base checkpoint not found: {base_best_ckpt}"
print("Using base checkpoint:", base_best_ckpt)

# GPU copy for evaluation / attacks
base = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
base.load_state_dict(torch.load(base_best_ckpt, map_location=device))
base.eval()
base_eval = base

# CPU copy for student generation
base_cpu = TransformerRegressor(in_feats=X_train.shape[-1]).to("cpu")
base_cpu.load_state_dict(torch.load(base_best_ckpt, map_location="cpu"))
base_cpu.eval()

print("Base model loaded on device and CPU.")

Using base checkpoint: /content/drive/MyDrive/298/Jena/proj_v3/baseModel/jena_base_best.pth
Base model loaded on device and CPU.


/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [ ]:
def model_n_params(model: nn.Module):
    return sum(p.numel() for p in model.parameters())

def model_file_size_mb(path: str):
    if not os.path.exists(path):
        return None
    return os.path.getsize(path) / (1024.0 * 1024.0)

def append_df_to_csv(df, path):
    header = not os.path.exists(path)
    df.to_csv(path, mode="a", header=header, index=False)

def is_enc_self_attn_param(name: str) -> bool:
    return name.startswith("encoder.layers.") and ".self_attn." in name

def is_ffn_param(name: str) -> bool:
    if not name.startswith("encoder.layers."):
        return False
    return (".linear1." in name) or (".linear2." in name)

def is_norm_param(name: str) -> bool:
    return name.startswith("encoder.layers.") and ".norm" in name

def is_inproj_head_param(name: str) -> bool:
    return name.startswith("in_proj") or name.startswith("head")

novel_student_specs = [
    dict(idx=5, name="student_05_enc_attn",    selector=is_enc_self_attn_param, noise_scale=1e-3),
    dict(idx=6, name="student_06_ffn",         selector=is_ffn_param,           noise_scale=1e-3),
    dict(idx=7, name="student_07_norms",       selector=is_norm_param,          noise_scale=1e-3),
    dict(idx=8, name="student_08_inproj_head", selector=is_inproj_head_param,   noise_scale=1e-3),
]

novel_student_paths = []

print("\n[JenaNov] creating novel students from base_cpu")

for spec in novel_student_specs:
    idx      = spec["idx"]
    name     = spec["name"]
    selector = spec["selector"]
    sigma    = spec["noise_scale"]

    print(f"  [Novel Student {idx}] {name} | sigma={sigma}")

    st = copy.deepcopy(base_cpu)

    with torch.no_grad():
        for pname, p in st.named_parameters():
            if selector(pname):
                p.add_(sigma * torch.randn_like(p))

    out_path = os.path.join(STUDENTS_NOV_DIR, f"{name}.pth")
    torch.save(st.state_dict(), out_path)
    novel_student_paths.append((idx, name, out_path))
    print("    saved to:", out_path)

    size_mb = model_file_size_mb(out_path)
    df_ms = pd.DataFrame([{
        "student_idx": idx,
        "model_name": name,
        "path": out_path,
        "params": model_n_params(base_cpu),
        "size_mb": size_mb,
    }])
    append_df_to_csv(df_ms, MODEL_SIZES_NOV_CSV)

novel_student_paths = sorted(novel_student_paths, key=lambda x: x[0])

print("\n[JenaNov] novel students generated:")
for idx, name, path in novel_student_paths:
    print(f"  {idx}: {name} - {path}")


[JenaNov] creating novel students from base_cpu
  [Novel Student 5] student_05_enc_attn | sigma=0.001
    saved to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/studentsNov/student_05_enc_attn.pth
  [Novel Student 6] student_06_ffn | sigma=0.001
    saved to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/studentsNov/student_06_ffn.pth
  [Novel Student 7] student_07_norms | sigma=0.001
    saved to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/studentsNov/student_07_norms.pth
  [Novel Student 8] student_08_inproj_head | sigma=0.001
    saved to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/studentsNov/student_08_inproj_head.pth

[JenaNov] novel students generated:
  5: student_05_enc_attn - /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/studentsNov/student_05_enc_attn.pth
  6: student_06_ffn - /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/studentsNov/student_06_ffn.pth
  7: student_07_norms - /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/studentsNov/student_07_norms.pth

In [ ]:
BIM_RANDOM_START_TRAIN = False
PGD_RANDOM_START_TRAIN = True
BIM_RANDOM_START_EVAL  = True
PGD_RANDOM_START_EVAL  = True

In [ ]:
def adv_train_student_fgsm_nov(
    student_path,
    out_dir,
    logs_dir,
    out_suffix="_fgsm_adv",
    epochs=FGSM_EPOCHS,
    lr=LR_STUDENT,
    eps_list=None,
    alpha=0.02,
    q=0.5,
):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = FGSM_TRAIN_EPS_LIST

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"fgsm_nov_adv_train_{student_stub}")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()

    monitor.start()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = rfgsm(model, xb, yb, eps=eps, alpha=alpha)
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)
                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                La_mean = torch.stack(adv_losses).mean() if len(adv_losses) > 0 else 0.0 * Lc
                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                L.backward()
                opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean += Lc.item() * bs
                tot_adv_mean += La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(
                f"[FGSM NOV train {student_stub}] "
                f"epoch {ep}/{epochs} | clean={clean_loss_epoch:.4f} "
                f"| adv_mean={adv_mean_epoch:.4f} | ep={ep_mins:.2f}m | total={total_mins:.2f}m"
            )

            ep_ckpt = os.path.join(out_dir, f"{student_stub}{out_suffix}_epoch{ep:02d}.pth")
            torch.save(model.to('cpu').state_dict(), ep_ckpt)
            model.to(device)

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    out = os.path.join(out_dir, os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth"))
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved FGSM-NOV adv student:", out)

    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved FGSM-NOV train history to:", hist_path)

    summary_row = monitor.summary()
    summary_row.update({
        "student": student_stub,
        "attack_train_type": "fgsm_nov",
        "epochs": epochs,
        "learning_rate": lr,
        "total_elapsed_minutes": (time.time() - start_time) / 60.0,
    })
    append_summary_row(summary_row, FGSM_NOV_TRAIN_RESOURCE_CSV)
    print("Updated FGSM NOV student training resource summary:", FGSM_NOV_TRAIN_RESOURCE_CSV)

    return out


def adv_train_student_bim_nov(
    student_path,
    out_dir,
    logs_dir,
    out_suffix="_bim_adv",
    epochs=BIM_EPOCHS,
    lr=LR_STUDENT,
    eps_list=None,
    iters=BIM_ITERS,
    q=0.5,
):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = BIM_TRAIN_EPS_LIST

    random_start = BIM_RANDOM_START_TRAIN

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"bim_nov_adv_train_{student_stub}")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()

    monitor.start()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = bim_attack(
                        model,
                        xb,
                        yb,
                        eps=eps,
                        alpha=eps / max(1, iters),
                        iters=iters,
                        random_start=random_start,
                    )
                    model.train()
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)
                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                La_mean = torch.stack(adv_losses).mean()
                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                L.backward()
                opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean += Lc.item() * bs
                tot_adv_mean += La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(
                f"[BIM NOV train {student_stub}] epoch {ep}/{epochs} "
                f"| clean={clean_loss_epoch:.4f} "
                f"| adv_mean={adv_mean_epoch:.4f} "
                f"| ep={ep_mins:.2f}m | total={total_mins:.2f}m"
            )

            ep_ckpt = os.path.join(out_dir, f"{student_stub}{out_suffix}_epoch{ep:02d}.pth")
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    out = os.path.join(out_dir, os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth"))
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved BIM NOV adv student:", out)

    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved BIM NOV train history to:", hist_path)

    summary_row = monitor.summary()
    summary_row.update({
        "student": student_stub,
        "attack_train_type": "bim_nov",
        "epochs": epochs,
        "learning_rate": lr,
        "iters": iters,
        "total_elapsed_minutes": (time.time() - start_time) / 60.0,
    })
    append_summary_row(summary_row, BIM_NOV_TRAIN_RESOURCE_CSV)
    print("Updated BIM NOV student training resource summary:", BIM_NOV_TRAIN_RESOURCE_CSV)

    return out


def adv_train_student_pgd_nov(
    student_path,
    out_dir,
    logs_dir,
    out_suffix="_pgd_adv",
    epochs=PGD_EPOCHS,
    lr=LR_STUDENT,
    eps_list=None,
    iters=PGD_ITERS,
    q=0.5,
):
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(logs_dir, exist_ok=True)

    if eps_list is None:
        eps_list = PGD_TRAIN_EPS_LIST

    random_start = PGD_RANDOM_START_TRAIN

    model = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
    model.load_state_dict(torch.load(student_path, map_location=device))
    opt = optim.AdamW(model.parameters(), lr=lr)

    student_stub = os.path.basename(student_path).replace(".pth", "")
    history = []

    hb_flag, hb_thread = start_heartbeat(f"pgd_nov_adv_train_{student_stub}")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()

    monitor.start()

    try:
        for ep in range(1, epochs + 1):
            model.train()
            tot_clean = 0.0
            tot_adv_mean = 0.0
            tot_by_eps = {eps: 0.0 for eps in eps_list}
            cnt = 0
            ep_start = time.time()

            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)

                pc = model(xb)
                Lc = rmse_loss(pc, yb) + quantile_loss(yb, pc, q)

                adv_losses = []
                adv_losses_per_eps = {}

                for eps in eps_list:
                    xa = pgd_attack(
                        model,
                        xb,
                        yb,
                        eps=eps,
                        alpha=eps / max(1, iters),
                        iters=iters,
                        random_start=random_start,
                    )
                    model.train()
                    pa = model(xa)
                    La = rmse_loss(pa, yb) + quantile_loss(yb, pa, q)
                    adv_losses.append(La)
                    adv_losses_per_eps[eps] = La

                La_mean = torch.stack(adv_losses).mean()
                L = 0.25 * Lc + 0.75 * La_mean

                opt.zero_grad()
                L.backward()
                opt.step()

                bs = xb.size(0)
                cnt += bs
                tot_clean += Lc.item() * bs
                tot_adv_mean += La_mean.item() * bs
                for eps in eps_list:
                    tot_by_eps[eps] += adv_losses_per_eps[eps].item() * bs

            ep_mins = (time.time() - ep_start) / 60.0
            total_mins = (time.time() - start_time) / 60.0
            clean_loss_epoch = tot_clean / max(1, cnt)
            adv_mean_epoch = tot_adv_mean / max(1, cnt)

            row = {
                "epoch": ep,
                "clean_loss": clean_loss_epoch,
                "adv_loss_mean": adv_mean_epoch,
                "epoch_minutes": ep_mins,
                "total_minutes": total_mins,
            }
            for eps in eps_list:
                row[f"adv_loss_eps_{eps:.2f}"] = tot_by_eps[eps] / max(1, cnt)
            history.append(row)

            print(
                f"[PGD NOV train {student_stub}] epoch {ep}/{epochs} "
                f"| clean={clean_loss_epoch:.4f} "
                f"| adv_mean={adv_mean_epoch:.4f} "
                f"| ep={ep_mins:.2f}m | total={total_mins:.2f}m"
            )

            ep_ckpt = os.path.join(out_dir, f"{student_stub}{out_suffix}_epoch{ep:02d}.pth")
            torch.save(model.to("cpu").state_dict(), ep_ckpt)
            model.to(device)

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    out = os.path.join(out_dir, os.path.basename(student_path).replace(".pth", f"{out_suffix}.pth"))
    torch.save(model.to("cpu").state_dict(), out)
    print("Saved PGD NOV adv student:", out)

    df_hist = pd.DataFrame(history)
    hist_path = os.path.join(logs_dir, f"{student_stub}{out_suffix}_train_history.csv")
    df_hist.to_csv(hist_path, index=False)
    print("Saved PGD NOV train history to:", hist_path)

    summary_row = monitor.summary()
    summary_row.update({
        "student": student_stub,
        "attack_train_type": "pgd_nov",
        "epochs": epochs,
        "learning_rate": lr,
        "iters": iters,
        "total_elapsed_minutes": (time.time() - start_time) / 60.0,
    })
    append_summary_row(summary_row, PGD_NOV_TRAIN_RESOURCE_CSV)
    print("Updated PGD NOV student training resource summary:", PGD_NOV_TRAIN_RESOURCE_CSV)

    return out

In [ ]:
FGSM_NOV_ADV_DIR = FGSM_NOV_STUDENTS_ADV_DIR
BIM_NOV_ADV_DIR  = BIM_NOV_STUDENTS_ADV_DIR
PGD_NOV_ADV_DIR  = PGD_NOV_STUDENTS_ADV_DIR

print("\n[JenaNov] Training FGSM adv NOV students")
nov_fgsm_adv_paths = []
for idx, name, sp in novel_student_paths:
    out = adv_train_student_fgsm_nov(
        sp,
        out_dir=FGSM_NOV_ADV_DIR,
        logs_dir=FGSM_NOV_LOGS_DIR,
        out_suffix="_fgsm_adv",
        epochs=FGSM_EPOCHS,
        lr=LR_STUDENT,
        eps_list=FGSM_TRAIN_EPS_LIST,
        alpha=0.02,
    )
    nov_fgsm_adv_paths.append((idx, name, out))

print("\n[JenaNov] Training BIM adv NOV students")
nov_bim_adv_paths = []
for idx, name, sp in novel_student_paths:
    out = adv_train_student_bim_nov(
        sp,
        out_dir=BIM_NOV_ADV_DIR,
        logs_dir=BIM_NOV_LOGS_DIR,
        out_suffix="_bim_adv",
        epochs=BIM_EPOCHS,
        lr=LR_STUDENT,
        eps_list=BIM_TRAIN_EPS_LIST,
        iters=BIM_ITERS,
    )
    nov_bim_adv_paths.append((idx, name, out))

print("\n[JenaNov] Training PGD adv NOV students")
nov_pgd_adv_paths = []
for idx, name, sp in novel_student_paths:
    out = adv_train_student_pgd_nov(
        sp,
        out_dir=PGD_NOV_ADV_DIR,
        logs_dir=PGD_NOV_LOGS_DIR,
        out_suffix="_pgd_adv",
        epochs=PGD_EPOCHS,
        lr=LR_STUDENT,
        eps_list=PGD_TRAIN_EPS_LIST,
        iters=PGD_ITERS,
    )
    nov_pgd_adv_paths.append((idx, name, out))

print("\n[JenaNov] Done adversarial training for NOV students.")


[JenaNov] Training FGSM adv NOV students


/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[FGSM NOV train student_05_enc_attn] epoch 1/5 | clean=0.0333 | adv_mean=0.0930 | ep=0.35m | total=0.35m
[FGSM NOV train student_05_enc_attn] epoch 2/5 | clean=0.0281 | adv_mean=0.0884 | ep=0.36m | total=0.71m
[FGSM NOV train student_05_enc_attn] epoch 3/5 | clean=0.0274 | adv_mean=0.0877 | ep=0.35m | total=1.06m
[FGSM NOV train student_05_enc_attn] epoch 4/5 | clean=0.0255 | adv_mean=0.0864 | ep=0.35m | total=1.41m
[FGSM NOV train student_05_enc_attn] epoch 5/5 | clean=0.0244 | adv_mean=0.0855 | ep=0.35m | total=1.76m
Saved FGSM-NOV adv student: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/fgsm/studentsNovAdv/student_05_enc_attn_fgsm_adv.pth
Saved FGSM-NOV train history to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/fgsm/results/train_logs/student_05_enc_attn_fgsm_adv_train_history.csv
Updated FGSM NOV student training resource summary: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/fgsm_nov_student_train_resource_summary.csv


/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[FGSM NOV train student_06_ffn] epoch 1/5 | clean=0.0332 | adv_mean=0.0928 | ep=0.36m | total=0.36m
[FGSM NOV train student_06_ffn] epoch 2/5 | clean=0.0294 | adv_mean=0.0891 | ep=0.36m | total=0.72m
[FGSM NOV train student_06_ffn] epoch 3/5 | clean=0.0264 | adv_mean=0.0873 | ep=0.35m | total=1.07m
[FGSM NOV train student_06_ffn] epoch 4/5 | clean=0.0259 | adv_mean=0.0866 | ep=0.35m | total=1.42m
[FGSM NOV train student_06_ffn] epoch 5/5 | clean=0.0249 | adv_mean=0.0855 | ep=0.36m | total=1.78m
Saved FGSM-NOV adv student: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/fgsm/studentsNovAdv/student_06_ffn_fgsm_adv.pth
Saved FGSM-NOV train history to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/fgsm/results/train_logs/student_06_ffn_fgsm_adv_train_history.csv
Updated FGSM NOV student training resource summary: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/fgsm_nov_student_train_resource_summary.csv


...

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[PGD NOV train student_07_norms] epoch 1/5 | clean=0.1313 | adv_mean=0.2093 | ep=1.60m | total=1.60m
[PGD NOV train student_07_norms] epoch 2/5 | clean=0.1036 | adv_mean=0.2016 | ep=1.62m | total=3.22m
[PGD NOV train student_07_norms] epoch 3/5 | clean=0.0780 | adv_mean=0.1773 | ep=1.67m | total=4.89m
[PGD NOV train student_07_norms] epoch 4/5 | clean=0.0644 | adv_mean=0.1653 | ep=1.62m | total=6.51m
[PGD NOV train student_07_norms] epoch 5/5 | clean=0.0497 | adv_mean=0.1698 | ep=1.65m | total=8.17m
Saved PGD NOV adv student: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/pgd/studentsNovAdv/student_07_norms_pgd_adv.pth
Saved PGD NOV train history to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/pgd/results/train_logs/student_07_norms_pgd_adv_train_history.csv
Updated PGD NOV student training resource summary: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/pgd_nov_student_train_resource_summary.csv


/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))
/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


[PGD NOV train student_08_inproj_head] epoch 1/5 | clean=0.1318 | adv_mean=0.2095 | ep=1.61m | total=1.61m
[PGD NOV train student_08_inproj_head] epoch 2/5 | clean=0.0732 | adv_mean=0.1985 | ep=1.62m | total=3.23m
[PGD NOV train student_08_inproj_head] epoch 3/5 | clean=0.0484 | adv_mean=0.1868 | ep=1.67m | total=4.90m
[PGD NOV train student_08_inproj_head] epoch 4/5 | clean=0.0425 | adv_mean=0.1759 | ep=1.62m | total=6.52m
[PGD NOV train student_08_inproj_head] epoch 5/5 | clean=0.0450 | adv_mean=0.1607 | ep=1.61m | total=8.12m
Saved PGD NOV adv student: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/pgd/studentsNovAdv/student_08_inproj_head_pgd_adv.pth
Saved PGD NOV train history to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/pgd/results/train_logs/student_08_inproj_head_pgd_adv_train_history.csv
Updated PGD NOV student training resource summary: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/pgd_nov_student_train_resource_summary.csv

[JenaNov] Done adve

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
def compute_transfer_matrix_nov(
    students,
    names,
    loader,
    attack_fn,
    eps_list,
    extra_kwargs_func,
    out_csv,
    attack_label="rfgsm_nov",
    trace_csv=None,
    summary_csv=None,
):
    rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_label}_transfer_matrix")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_time = time.time()

    monitor.start()

    try:
        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)
            print(f"\n[{attack_label.upper()}] Transfer matrix for eps={eps}...")

            for i, atk_name in enumerate(names):
                for j, def_name in enumerate(names):
                    atk_model = students[i]
                    def_model = students[j]

                    rmse_ij = eval_pair_attack(
                        defender=def_model,
                        attacker=atk_model,
                        loader=loader,
                        attack_fn=attack_fn,
                        atk_kwargs=atk_kwargs,
                    )

                    rows.append({
                        "attack": attack_label,
                        "epsilon": eps,
                        "attacker": atk_name,
                        "defender": def_name,
                        "rmse_adv": rmse_ij,
                    })

                    print(
                        f"  atk={atk_name} - def={def_name} | "
                        f"eps={eps:.2f} | RMSE={rmse_ij:.5f}"
                    )
    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"\nSaved {attack_label} transfer matrix to:", out_csv)

    if trace_csv is not None and summary_csv is not None:
        save_monitor_outputs(
            monitor,
            trace_csv=trace_csv,
            summary_csv=summary_csv,
            extra_summary={
                "stage": f"{attack_label}_transfer_matrix",
                "total_elapsed_minutes": (time.time() - start_time) / 60.0,
            }
        )

    return df

In [ ]:
def run_morphence_eval_single_pass_nov(
    attack_name,
    attack_fn,
    eps_list,
    students,
    results_csv,
    stats_csv,
    extra_kwargs_func,
    trace_csv,
    summary_csv,
):
    all_rows = []

    hb_flag, hb_thread = start_heartbeat(f"{attack_name}_single_pass_eval")
    monitor = ResourceMonitor(sample_interval=2.0)
    start_all = time.time()

    monitor.start()

    try:
        base_clean = eval_clean_rmse(base_eval, test_dl)
        morph_clean = eval_ensemble_rmse(students, test_dl)

        all_rows.append(["clean", None, "base", base_clean, 1, SEED])
        all_rows.append(["clean", None, "morph_ensemble", morph_clean, 1, SEED])

        print(
            f"[{attack_name.upper()}] clean | "
            f"base={base_clean:.5f} | morph_ensemble={morph_clean:.5f}"
        )

        for eps in eps_list:
            atk_kwargs = extra_kwargs_func(eps)

            base_rmse = eval_pair_attack(
                defender=base_eval,
                attacker=base_eval,
                loader=test_dl,
                attack_fn=attack_fn,
                atk_kwargs=atk_kwargs,
            )

            def atk_kwargs_wrapper():
                return atk_kwargs

            morph_rmse = eval_random_switch_attack(
                models=students,
                loader=test_dl,
                attack_fn=attack_fn,
                atk_kwargs_func=atk_kwargs_wrapper,
            )

            all_rows.append([attack_name, eps, "base", base_rmse, 1, SEED])
            all_rows.append([attack_name, eps, "morph_rand", morph_rmse, 1, SEED])

            print(
                f"[{attack_name.upper()}] eps={eps:.2f} | "
                f"base={base_rmse:.5f} | morph_rand={morph_rmse:.5f}"
            )

    finally:
        monitor.stop()
        stop_heartbeat(hb_flag, hb_thread)

    df_runs = pd.DataFrame(
        all_rows,
        columns=["attack", "epsilon", "model", "RMSE", "run", "seed"]
    )
    df_runs.to_csv(results_csv, index=False)

    stats = (
        df_runs
        .groupby(["attack", "epsilon", "model"])["RMSE"]
        .agg(mean="mean", std="std", n="count")
        .reset_index()
        .sort_values(["attack", "epsilon", "model"])
    )
    stats.to_csv(stats_csv, index=False)

    save_monitor_outputs(
        monitor,
        trace_csv=trace_csv,
        summary_csv=summary_csv,
        extra_summary={
            "stage": f"{attack_name}_single_pass_eval",
            "total_elapsed_minutes": (time.time() - start_all) / 60.0,
        }
    )

    print(f"\nSaved {attack_name} runs to:", results_csv)
    print(f"Saved {attack_name} stats to:", stats_csv)

    return df_runs, stats

In [ ]:
def load_nov_adv_students(path_list):
    students = []
    for p in path_list:
        m = TransformerRegressor(in_feats=X_train.shape[-1]).to(device)
        m.load_state_dict(torch.load(p, map_location=device))
        m.eval()
        students.append(m)
    return students

fgsm_nov_adv_paths = sorted([
    os.path.join(FGSM_NOV_STUDENTS_ADV_DIR, f)
    for f in os.listdir(FGSM_NOV_STUDENTS_ADV_DIR)
    if f.endswith("_fgsm_adv.pth")
])

bim_nov_adv_paths = sorted([
    os.path.join(BIM_NOV_STUDENTS_ADV_DIR, f)
    for f in os.listdir(BIM_NOV_STUDENTS_ADV_DIR)
    if f.endswith("_bim_adv.pth")
])

pgd_nov_adv_paths = sorted([
    os.path.join(PGD_NOV_STUDENTS_ADV_DIR, f)
    for f in os.listdir(PGD_NOV_STUDENTS_ADV_DIR)
    if f.endswith("_pgd_adv.pth")
])

fgsm_nov_students = load_nov_adv_students(fgsm_nov_adv_paths)
bim_nov_students  = load_nov_adv_students(bim_nov_adv_paths)
pgd_nov_students  = load_nov_adv_students(pgd_nov_adv_paths)

fgsm_nov_student_names = [os.path.basename(p).replace(".pth", "") for p in fgsm_nov_adv_paths]
bim_nov_student_names  = [os.path.basename(p).replace(".pth", "") for p in bim_nov_adv_paths]
pgd_nov_student_names  = [os.path.basename(p).replace(".pth", "") for p in pgd_nov_adv_paths]

print("\nLoaded NOV adv student pools:")
print(" FGSM NOV:", len(fgsm_nov_students))
print(" BIM  NOV:", len(bim_nov_students))
print(" PGD  NOV:", len(pgd_nov_students))


Loaded NOV adv student pools:
 FGSM NOV: 4
 BIM  NOV: 4
 PGD  NOV: 4


/tmp/ipykernel_4143/2463783413.py:34: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


In [ ]:
print("\n[FGSM-Nov] diversity + transferability + single-pass eval")

def fgsm_nov_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps,
    }

_ = compute_weight_diversity(
    fgsm_nov_students,
    fgsm_nov_student_names,
    NOV_FGSM_WEIGHT_DIVERSITY,
)

_ = compute_prediction_diversity(
    fgsm_nov_students,
    test_dl,
    NOV_FGSM_PRED_DIVERSITY,
)

_ = compute_transfer_matrix_nov(
    students=fgsm_nov_students,
    names=fgsm_nov_student_names,
    loader=test_dl,
    attack_fn=rfgsm,
    eps_list=EPS_LIST,
    extra_kwargs_func=fgsm_nov_kwargs,
    out_csv=NOV_FGSM_XFER_MATRIX_CSV,
    attack_label="rfgsm_nov",
    trace_csv=FGSM_NOV_XFER_TRACE_CSV,
    summary_csv=FGSM_NOV_XFER_SUMMARY_CSV,
)

df_fgsm_nov_runs, fgsm_nov_stats = run_morphence_eval_single_pass_nov(
    attack_name="rfgsm_nov",
    attack_fn=rfgsm,
    eps_list=EPS_LIST,
    students=fgsm_nov_students,
    results_csv=NOV_FGSM_RUNS_CSV,
    stats_csv=NOV_FGSM_STATS_CSV,
    extra_kwargs_func=fgsm_nov_kwargs,
    trace_csv=FGSM_NOV_EVAL_TRACE_CSV,
    summary_csv=FGSM_NOV_EVAL_SUMMARY_CSV,
)


[FGSM-Nov] diversity + transferability + single-pass eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/fgsm/results/nov_fgsm_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/fgsm/results/nov_fgsm_prediction_diversity.csv

[RFGSM_NOV] Transfer matrix for eps=0.1...
  atk=student_05_enc_attn_fgsm_adv - def=student_05_enc_attn_fgsm_adv | eps=0.10 | RMSE=0.11754
  atk=student_05_enc_attn_fgsm_adv - def=student_06_ffn_fgsm_adv | eps=0.10 | RMSE=0.11096
  atk=student_05_enc_attn_fgsm_adv - def=student_07_norms_fgsm_adv | eps=0.10 | RMSE=0.11434
  atk=student_05_enc_attn_fgsm_adv - def=student_08_inproj_head_fgsm_adv | eps=0.10 | RMSE=0.11469
  atk=student_06_ffn_fgsm_adv - def=student_05_enc_attn_fgsm_adv | eps=0.10 | RMSE=0.11566
  atk=student_06_ffn_fgsm_adv - def=student_06_ffn_fgsm_adv | eps=0.10 | RMSE=0.11340
  atk=student_06_ffn_fgsm_adv - def=student_07_norms_fgsm_adv | eps=0.10 | RMSE=0.11408
  atk=stu

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[RFGSM_NOV] clean | base=0.01484 | morph_ensemble=0.01655
[RFGSM_NOV] eps=0.10 | base=0.13989 | morph_rand=0.11377
[RFGSM_NOV] eps=0.20 | base=0.22672 | morph_rand=0.20775
[RFGSM_NOV] eps=0.30 | base=0.28999 | morph_rand=0.26847
[RFGSM_NOV] eps=0.50 | base=0.33464 | morph_rand=0.33176
Saved resource trace to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/fgsm_nov_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/fgsm_nov_eval_resource_summary.csv
Resource summary: {'n_samples': 5, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1840.6953125, 'max_gpu_mem_mb': 1035.0, 'avg_gpu_util_percent': 30.0, 'gpu_active_percent_of_samples': 80.0, 'est_gpu_energy_j': 1151.0659577941894, 'stage': 'rfgsm_nov_single_pass_eval', 'total_elapsed_minutes': 0.1848773161570231}

Saved rfgsm_nov runs to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/fgsm/results/nov_fgsm_morphence_runs_single.csv
Saved rfgsm_nov st

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
print("\n[BIM-Nov] diversity + transferability + single-pass eval")

def bim_nov_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / BIM_ITERS,
        "iters": BIM_ITERS,
        "random_start": True,
    }

_ = compute_weight_diversity(
    bim_nov_students,
    bim_nov_student_names,
    NOV_BIM_WEIGHT_DIVERSITY,
)

_ = compute_prediction_diversity(
    bim_nov_students,
    test_dl,
    NOV_BIM_PRED_DIVERSITY,
)

_ = compute_transfer_matrix_nov(
    students=bim_nov_students,
    names=bim_nov_student_names,
    loader=test_dl,
    attack_fn=bim_attack,
    eps_list=EPS_LIST,
    extra_kwargs_func=bim_nov_kwargs,
    out_csv=NOV_BIM_XFER_MATRIX_CSV,
    attack_label="bim_nov",
    trace_csv=BIM_NOV_XFER_TRACE_CSV,
    summary_csv=BIM_NOV_XFER_SUMMARY_CSV,
)

df_bim_nov_runs, bim_nov_stats = run_morphence_eval_single_pass_nov(
    attack_name="bim_nov",
    attack_fn=bim_attack,
    eps_list=EPS_LIST,
    students=bim_nov_students,
    results_csv=NOV_BIM_RUNS_CSV,
    stats_csv=NOV_BIM_STATS_CSV,
    extra_kwargs_func=bim_nov_kwargs,
    trace_csv=BIM_NOV_EVAL_TRACE_CSV,
    summary_csv=BIM_NOV_EVAL_SUMMARY_CSV,
)


[BIM-Nov] diversity + transferability + single-pass eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/bim/results/nov_bim_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/bim/results/nov_bim_prediction_diversity.csv

[BIM_NOV] Transfer matrix for eps=0.1...
  atk=student_05_enc_attn_bim_adv - def=student_05_enc_attn_bim_adv | eps=0.10 | RMSE=0.15141
  atk=student_05_enc_attn_bim_adv - def=student_06_ffn_bim_adv | eps=0.10 | RMSE=0.14203
  atk=student_05_enc_attn_bim_adv - def=student_07_norms_bim_adv | eps=0.10 | RMSE=0.13922
  atk=student_05_enc_attn_bim_adv - def=student_08_inproj_head_bim_adv | eps=0.10 | RMSE=0.15164
  atk=student_06_ffn_bim_adv - def=student_05_enc_attn_bim_adv | eps=0.10 | RMSE=0.15140
  atk=student_06_ffn_bim_adv - def=student_06_ffn_bim_adv | eps=0.10 | RMSE=0.14203
  atk=student_06_ffn_bim_adv - def=student_07_norms_bim_adv | eps=0.10 | RMSE=0.13921
  atk=student_06_ffn_bim_adv -

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[BIM_NOV] clean | base=0.01484 | morph_ensemble=0.14465
[BIM_NOV] eps=0.10 | base=0.12691 | morph_rand=0.14743
[BIM_NOV] eps=0.20 | base=0.21686 | morph_rand=0.14433
[BIM_NOV] eps=0.30 | base=0.29850 | morph_rand=0.14778
[BIM_NOV] eps=0.50 | base=0.42823 | morph_rand=0.14596
Saved resource trace to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/bim_nov_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/bim_nov_eval_resource_summary.csv
Resource summary: {'n_samples': 25, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1840.84375, 'max_gpu_mem_mb': 1035.0, 'avg_gpu_util_percent': 45.28, 'gpu_active_percent_of_samples': 96.0, 'est_gpu_energy_j': 7814.013065599203, 'stage': 'bim_nov_single_pass_eval', 'total_elapsed_minutes': 0.8566637635231018}

Saved bim_nov runs to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/bim/results/nov_bim_morphence_runs_single.csv
Saved bim_nov stats to: /content/driv

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
print("\n[PGD-Nov] diversity + transferability + single-pass eval")

def pgd_nov_kwargs(eps):
    return {
        "eps": eps,
        "alpha": eps / PGD_ITERS,
        "iters": PGD_ITERS,
        "random_start": True,
    }

_ = compute_weight_diversity(
    pgd_nov_students,
    pgd_nov_student_names,
    NOV_PGD_WEIGHT_DIVERSITY,
)

_ = compute_prediction_diversity(
    pgd_nov_students,
    test_dl,
    NOV_PGD_PRED_DIVERSITY,
)

_ = compute_transfer_matrix_nov(
    students=pgd_nov_students,
    names=pgd_nov_student_names,
    loader=test_dl,
    attack_fn=pgd_attack,
    eps_list=EPS_LIST,
    extra_kwargs_func=pgd_nov_kwargs,
    out_csv=NOV_PGD_XFER_MATRIX_CSV,
    attack_label="pgd_nov",
    trace_csv=PGD_NOV_XFER_TRACE_CSV,
    summary_csv=PGD_NOV_XFER_SUMMARY_CSV,
)

df_pgd_nov_runs, pgd_nov_stats = run_morphence_eval_single_pass_nov(
    attack_name="pgd_nov",
    attack_fn=pgd_attack,
    eps_list=EPS_LIST,
    students=pgd_nov_students,
    results_csv=NOV_PGD_RUNS_CSV,
    stats_csv=NOV_PGD_STATS_CSV,
    extra_kwargs_func=pgd_nov_kwargs,
    trace_csv=PGD_NOV_EVAL_TRACE_CSV,
    summary_csv=PGD_NOV_EVAL_SUMMARY_CSV,
)


[PGD-Nov] diversity + transferability + single-pass eval
Saved weight diversity to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/pgd/results/nov_pgd_weight_diversity.csv
Saved prediction diversity to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/pgd/results/nov_pgd_prediction_diversity.csv

[PGD_NOV] Transfer matrix for eps=0.1...
  atk=student_05_enc_attn_pgd_adv - def=student_05_enc_attn_pgd_adv | eps=0.10 | RMSE=0.12679
  atk=student_05_enc_attn_pgd_adv - def=student_06_ffn_pgd_adv | eps=0.10 | RMSE=0.08961
  atk=student_05_enc_attn_pgd_adv - def=student_07_norms_pgd_adv | eps=0.10 | RMSE=0.10721
  atk=student_05_enc_attn_pgd_adv - def=student_08_inproj_head_pgd_adv | eps=0.10 | RMSE=0.09682
  atk=student_06_ffn_pgd_adv - def=student_05_enc_attn_pgd_adv | eps=0.10 | RMSE=0.09672
  atk=student_06_ffn_pgd_adv - def=student_06_ffn_pgd_adv | eps=0.10 | RMSE=0.10655
  atk=student_06_ffn_pgd_adv - def=student_07_norms_pgd_adv | eps=0.10 | RMSE=0.09392
  atk=student_06_ffn_pgd_adv -

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


[PGD_NOV] clean | base=0.01484 | morph_ensemble=0.02411
[PGD_NOV] eps=0.10 | base=0.12694 | morph_rand=0.10154
[PGD_NOV] eps=0.20 | base=0.21728 | morph_rand=0.13546
[PGD_NOV] eps=0.30 | base=0.29857 | morph_rand=0.15336
[PGD_NOV] eps=0.50 | base=0.42856 | morph_rand=0.16048
Saved resource trace to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/pgd_nov_eval_resource_trace.csv
Saved resource summary to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/pgd_nov_eval_resource_summary.csv
Resource summary: {'n_samples': 26, 'sample_interval_sec': 2.0, 'max_cpu_ram_mb': 1840.96484375, 'max_gpu_mem_mb': 1035.0, 'avg_gpu_util_percent': 43.80769230769231, 'gpu_active_percent_of_samples': 96.15384615384616, 'est_gpu_energy_j': 8181.483447651863, 'stage': 'pgd_nov_single_pass_eval', 'total_elapsed_minutes': 0.8902441024780273}

Saved pgd_nov runs to: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/pgd/results/nov_pgd_morphence_runs_single.csv
Saved pgd

/tmp/ipykernel_4143/2403520972.py:106: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  est_gpu_energy_j = float(np.trapz(p[mask], t[mask]))


In [ ]:
print("\n================ JENANOV OUTPUTS ================\n")

print("[FGSM-Nov] runs               :", NOV_FGSM_RUNS_CSV)
print("[BIM-Nov]  runs               :", NOV_BIM_RUNS_CSV)
print("[PGD-Nov]  runs               :", NOV_PGD_RUNS_CSV)

print("[FGSM-Nov] weight diversity   :", NOV_FGSM_WEIGHT_DIVERSITY)
print("[BIM-Nov]  weight diversity   :", NOV_BIM_WEIGHT_DIVERSITY)
print("[PGD-Nov]  weight diversity   :", NOV_PGD_WEIGHT_DIVERSITY)

print("[FGSM-Nov] prediction diversity:", NOV_FGSM_PRED_DIVERSITY)
print("[BIM-Nov]  prediction diversity:", NOV_BIM_PRED_DIVERSITY)
print("[PGD-Nov]  prediction diversity:", NOV_PGD_PRED_DIVERSITY)

print("[FGSM-Nov] transfer matrix    :", NOV_FGSM_XFER_MATRIX_CSV)
print("[BIM-Nov]  transfer matrix    :", NOV_BIM_XFER_MATRIX_CSV)
print("[PGD-Nov]  transfer matrix    :", NOV_PGD_XFER_MATRIX_CSV)

print("\n================ JENANOV COMPUTATIONAL METRICS ================\n")

print("Model sizes CSV               :", MODEL_SIZES_NOV_CSV)

print("FGSM NOV train summary        :", FGSM_NOV_TRAIN_RESOURCE_CSV)
print("BIM  NOV train summary        :", BIM_NOV_TRAIN_RESOURCE_CSV)
print("PGD  NOV train summary        :", PGD_NOV_TRAIN_RESOURCE_CSV)

print("FGSM NOV transfer trace       :", FGSM_NOV_XFER_TRACE_CSV)
print("FGSM NOV transfer summary     :", FGSM_NOV_XFER_SUMMARY_CSV)
print("BIM  NOV transfer trace       :", BIM_NOV_XFER_TRACE_CSV)
print("BIM  NOV transfer summary     :", BIM_NOV_XFER_SUMMARY_CSV)
print("PGD  NOV transfer trace       :", PGD_NOV_XFER_TRACE_CSV)
print("PGD  NOV transfer summary     :", PGD_NOV_XFER_SUMMARY_CSV)

print("FGSM NOV eval trace           :", FGSM_NOV_EVAL_TRACE_CSV)
print("FGSM NOV eval summary         :", FGSM_NOV_EVAL_SUMMARY_CSV)
print("BIM  NOV eval trace           :", BIM_NOV_EVAL_TRACE_CSV)
print("BIM  NOV eval summary         :", BIM_NOV_EVAL_SUMMARY_CSV)
print("PGD  NOV eval trace           :", PGD_NOV_EVAL_TRACE_CSV)
print("PGD  NOV eval summary         :", PGD_NOV_EVAL_SUMMARY_CSV)


================ JENANOV OUTPUTS ================

[FGSM-Nov] runs               : /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/fgsm/results/nov_fgsm_morphence_runs_single.csv
[BIM-Nov]  runs               : /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/bim/results/nov_bim_morphence_runs_single.csv
[PGD-Nov]  runs               : /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/pgd/results/nov_pgd_morphence_runs_single.csv
[FGSM-Nov] weight diversity   : /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/fgsm/results/nov_fgsm_weight_diversity.csv
[BIM-Nov]  weight diversity   : /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/bim/results/nov_bim_weight_diversity.csv
[PGD-Nov]  weight diversity   : /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/pgd/results/nov_pgd_weight_diversity.csv
[FGSM-Nov] prediction diversity: /content/drive/MyDrive/298/Jena/proj_v3/jenaNov/fgsm/results/nov_fgsm_prediction_diversity.csv
[BIM-Nov]  prediction diversity: /content/drive/MyDrive/298/Jena/proj_v3/je

In [ ]:
# FINAL COLAB CLEANUP CELL
# Put this as the very last cell in the notebook.

import gc
import time
import torch
from google.colab import runtime

print("Final cleanup starting...")

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(10)

print("Disconnecting and deleting runtime now...")
runtime.unassign()

Final cleanup starting...
Disconnecting and deleting runtime now...


In [ ]:
# =========================================================
# FULLY SELF-CONTAINED COMPUTATIONAL METRICS REPORT
# (SAFE FOR NEW COLAB NOTEBOOK)
# =========================================================

# -----------------------------
# STEP 0: Mount Google Drive
# -----------------------------
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except:
    print("⚠️ Not running in Colab? Make sure paths are accessible.")

# -----------------------------
# imports
# -----------------------------
import os
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

# -----------------------------
# base paths
# -----------------------------
BASE_PROJ_DIR = "/content/drive/MyDrive/298/Jena/proj_v3"
COMP_METRICS_DIR = os.path.join(BASE_PROJ_DIR, "computational_metrics")

JENANOV_DIR = os.path.join(BASE_PROJ_DIR, "jenaNov")
JENANOV_COMP_METRICS_DIR = os.path.join(JENANOV_DIR, "computational_metrics")

# -----------------------------
# sanity check
# -----------------------------
if not os.path.exists(BASE_PROJ_DIR):
    raise FileNotFoundError(
        f"❌ Base directory not found:\n{BASE_PROJ_DIR}\n"
        "Make sure Google Drive is mounted and the path is correct."
    )
else:
    print("✅ Drive mounted and base directory found.")

# -----------------------------
# helper functions
# -----------------------------
def safe_read_csv(path):
    if os.path.exists(path):
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f"⚠️ Could not read {path}: {e}")
            return None
    else:
        print(f"⚠️ Missing file: {path}")
    return None

def add_source_columns(df, section, metric_name, source_path):
    if df is None or len(df) == 0:
        return None
    df = df.copy()
    df.insert(0, "section", section)
    df.insert(1, "metric_name", metric_name)
    df.insert(2, "source_csv", source_path)
    return df

def full_csv(path, section, metric_name):
    df = safe_read_csv(path)
    if df is None or len(df) == 0:
        return None
    return add_source_columns(df, section, metric_name, path)

def show_table(title, df, sort_cols=None):
    display(Markdown(f"## {title}"))
    if df is None or len(df) == 0:
        display(Markdown("_No data found._"))
        return
    out = df.copy()
    if sort_cols is not None:
        existing = [c for c in sort_cols if c in out.columns]
        if existing:
            out = out.sort_values(existing).reset_index(drop=True)
    display(out)

# -----------------------------
# file definitions
# -----------------------------
files_summary = [
    # base
    ("Original", "Base training summary", os.path.join(COMP_METRICS_DIR, "base_resource_summary.csv")),
    ("Original", "Base model profile", os.path.join(COMP_METRICS_DIR, "base_model_profile.csv")),

    # student training
    ("Original", "FGSM student train summary", os.path.join(COMP_METRICS_DIR, "fgsm_student_train_resource_summary.csv")),
    ("Original", "BIM student train summary",  os.path.join(COMP_METRICS_DIR, "bim_student_train_resource_summary.csv")),
    ("Original", "PGD student train summary",  os.path.join(COMP_METRICS_DIR, "pgd_student_train_resource_summary.csv")),

    # transfer
    ("Original", "FGSM transfer summary", os.path.join(COMP_METRICS_DIR, "fgsm_transfer_resource_summary.csv")),
    ("Original", "BIM transfer summary",  os.path.join(COMP_METRICS_DIR, "bim_transfer_resource_summary.csv")),
    ("Original", "PGD transfer summary",  os.path.join(COMP_METRICS_DIR, "pgd_transfer_resource_summary.csv")),

    # eval
    ("Original", "FGSM eval summary", os.path.join(COMP_METRICS_DIR, "fgsm_eval_resource_summary.csv")),
    ("Original", "BIM eval summary",  os.path.join(COMP_METRICS_DIR, "bim_eval_resource_summary.csv")),
    ("Original", "PGD eval summary",  os.path.join(COMP_METRICS_DIR, "pgd_eval_resource_summary.csv")),

    # novel
    ("JenaNov", "Model sizes", os.path.join(JENANOV_DIR, "jenaNov_model_sizes.csv")),

    ("JenaNov", "FGSM NOV student train summary", os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_student_train_resource_summary.csv")),
    ("JenaNov", "BIM NOV student train summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_student_train_resource_summary.csv")),
    ("JenaNov", "PGD NOV student train summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_student_train_resource_summary.csv")),

    ("JenaNov", "FGSM NOV transfer summary", os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_transfer_resource_summary.csv")),
    ("JenaNov", "BIM NOV transfer summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_transfer_resource_summary.csv")),
    ("JenaNov", "PGD NOV transfer summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_transfer_resource_summary.csv")),

    ("JenaNov", "FGSM NOV eval summary", os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_eval_resource_summary.csv")),
    ("JenaNov", "BIM NOV eval summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_eval_resource_summary.csv")),
    ("JenaNov", "PGD NOV eval summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_eval_resource_summary.csv")),
]

# -----------------------------
# load everything
# -----------------------------
all_tables = []
for section, metric_name, path in files_summary:
    df = full_csv(path, section, metric_name)
    if df is not None:
        all_tables.append(df)

combined = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()

# -----------------------------
# output
# -----------------------------
display(Markdown("# 📊 Computational Metrics Summary"))

show_table(
    "All Metrics Combined",
    combined,
    sort_cols=["section", "metric_name", "student", "attack_train_type", "stage"]
)

# grouped views
if len(combined) > 0:
    show_table(
        "Base / Model Profiles",
        combined[combined["metric_name"].str.contains("Base|Model sizes", na=False)],
        sort_cols=["section", "metric_name"]
    )

    show_table(
        "Training Metrics",
        combined[combined["metric_name"].str.contains("train summary", case=False, na=False)],
        sort_cols=["section", "metric_name", "student"]
    )

    show_table(
        "Transfer Metrics",
        combined[combined["metric_name"].str.contains("transfer summary", case=False, na=False)],
        sort_cols=["section", "metric_name"]
    )

    show_table(
        "Evaluation Metrics",
        combined[combined["metric_name"].str.contains("eval summary", case=False, na=False)],
        sort_cols=["section", "metric_name"]
    )

# compact view
display(Markdown("## 📌 Compact Table"))

compact_cols = [
    "section", "metric_name", "student", "attack_train_type", "stage",
    "epochs", "learning_rate", "iters", "n_samples",
    "max_cpu_ram_mb", "max_gpu_mem_mb",
    "avg_gpu_util_percent", "gpu_active_percent_of_samples",
    "est_gpu_energy_j", "total_elapsed_minutes",
    "estimated_macs", "estimated_flops", "parameter_count"
]

if len(combined) > 0:
    compact_cols = [c for c in compact_cols if c in combined.columns]
    display(combined[compact_cols])
else:
    display(Markdown("_No data found._"))

Mounted at /content/drive
✅ Drive mounted and base directory found.


# 📊 Computational Metrics Summary

## All Metrics Combined

,section,metric_name,source_csv,n_samples,sample_interval_sec,max_cpu_ram_mb,max_gpu_mem_mb,avg_gpu_util_percent,gpu_active_percent_of_samples,est_gpu_energy_j,stage,total_elapsed_minutes,epochs,learning_rate,model,estimated_macs,estimated_flops,parameter_count,student,attack_train_type,iters,student_idx,model_name,path,params,size_mb
0,JenaNov,BIM NOV eval summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/bim_nov_eval_resource_summary.csv,25.0,2.0,1840.843750,1035.0,45.280000,96.000000,7814.013066,bim_nov_single_pass_eval,0.856664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JenaNov,BIM NOV student train summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/bim_nov_student_train_resource_summary.csv,242.0,2.0,1836.347656,1035.0,47.706612,99.586777,81132.252779,NaN,8.148818,5.0,0.001,NaN,NaN,NaN,NaN,student_05_enc_attn,bim_nov,10.0,NaN,NaN,NaN,NaN,NaN
2,JenaNov,BIM NOV student train summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/bim_nov_student_train_resource_summary.csv,246.0,2.0,1836.351562,1035.0,46.239837,99.593496,72495.282096,NaN,8.283573,5.0,0.001,NaN,NaN,NaN,NaN,student_06_ffn,bim_nov,10.0,NaN,NaN,NaN,NaN,NaN
3,JenaNov,BIM NOV student train summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/bim_nov_student_train_resource_summary.csv,242.0,2.0,1836.347656,1035.0,46.809917,99.586777,69954.705060,NaN,8.149100,5.0,0.001,NaN,NaN,NaN,NaN,student_07_norms,bim_nov,10.0,NaN,NaN,NaN,NaN,NaN
4,JenaNov,BIM NOV student train summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/bim_nov_student_train_resource_summary.csv,245.0,2.0,1836.347656,1035.0,46.159184,99.591837,71087.125747,NaN,8.250743,5.0,0.001,NaN,NaN,NaN,NaN,student_08_inproj_head,bim_nov,10.0,NaN,NaN,NaN,NaN,NaN
5,JenaNov,BIM NOV transfer summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/bim_nov_transfer_resource_summary.csv,197.0,2.0,1840.824219,1035.0,46.121827,100.000000,65096.223682,bim_nov_transfer_matrix,6.736096,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,JenaNov,FGSM NOV eval summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/fgsm_nov_eval_resource_summary.csv,5.0,2.0,1840.695312,1035.0,30.000000,80.000000,1151.065958,rfgsm_nov_single_pass_eval,0.184877,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,JenaNov,FGSM NOV student train summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/fgsm_nov_student_train_resource_summary.csv,42.0,2.0,1836.269531,1035.0,44.976190,97.619048,15852.609263,NaN,1.780206,5.0,0.001,NaN,NaN,NaN,NaN,student_05_enc_attn,fgsm_nov,NaN,NaN,NaN,NaN,NaN,NaN
8,JenaNov,FGSM NOV student train summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/fgsm_nov_student_train_resource_summary.csv,43.0,2.0,1836.285156,1035.0,44.744186,97.674419,16525.805460,NaN,1.819161,5.0,0.001,NaN,NaN,NaN,NaN,student_06_ffn,fgsm_nov,NaN,NaN,NaN,NaN,NaN,NaN
9,JenaNov,FGSM NOV student train summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/fgsm_nov_student_train_resource_summary.csv,53.0,2.0,1836.296875,1035.0,45.132075,98.113208,17252.633300,NaN,1.798459,5.0,0.001,NaN,NaN,NaN,NaN,student_07_norms,fgsm_nov,NaN,NaN,NaN,NaN,NaN,NaN


## Base / Model Profiles

...

,section,metric_name,source_csv,n_samples,sample_interval_sec,max_cpu_ram_mb,max_gpu_mem_mb,avg_gpu_util_percent,gpu_active_percent_of_samples,est_gpu_energy_j,stage,total_elapsed_minutes,epochs,learning_rate,model,estimated_macs,estimated_flops,parameter_count,student,attack_train_type,iters,student_idx,model_name,path,params,size_mb
0,JenaNov,BIM NOV transfer summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/bim_nov_transfer_resource_summary.csv,197.0,2.0,1840.824219,1035.0,46.121827,100.0,65096.223682,bim_nov_transfer_matrix,6.736096,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JenaNov,FGSM NOV transfer summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/fgsm_nov_transfer_resource_summary.csv,31.0,2.0,1840.671875,1035.0,37.774194,100.0,9341.488310,rfgsm_nov_transfer_matrix,1.073917,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JenaNov,PGD NOV transfer summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/pgd_nov_transfer_resource_summary.csv,200.0,2.0,1840.964844,1035.0,46.530000,100.0,64584.341816,pgd_nov_transfer_matrix,6.735098,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Original,BIM transfer summary,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/bim_transfer_resource_summary.csv,199.0,2.0,1835.156250,977.0,46.437186,100.0,64818.306245,bim_transfer_matrix,6.700962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Original,FGSM transfer summary,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/fgsm_transfer_resource_summary.csv,30.0,2.0,1834.300781,977.0,38.466667,100.0,8601.731845,rfgsm_transfer_matrix,1.038785,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Original,PGD transfer summary,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/pgd_transfer_resource_summary.csv,191.0,2.0,1835.234375,977.0,46.471204,100.0,65766.544961,pgd_transfer_matrix,6.744692,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Evaluation Metrics

,section,metric_name,source_csv,n_samples,sample_interval_sec,max_cpu_ram_mb,max_gpu_mem_mb,avg_gpu_util_percent,gpu_active_percent_of_samples,est_gpu_energy_j,stage,total_elapsed_minutes,epochs,learning_rate,model,estimated_macs,estimated_flops,parameter_count,student,attack_train_type,iters,student_idx,model_name,path,params,size_mb
0,JenaNov,BIM NOV eval summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/bim_nov_eval_resource_summary.csv,25.0,2.0,1840.843750,1035.0,45.280000,96.000000,7814.013066,bim_nov_single_pass_eval,0.856664,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,JenaNov,FGSM NOV eval summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/fgsm_nov_eval_resource_summary.csv,5.0,2.0,1840.695312,1035.0,30.000000,80.000000,1151.065958,rfgsm_nov_single_pass_eval,0.184877,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,JenaNov,PGD NOV eval summary,/content/drive/MyDrive/298/Jena/proj_v3/jenaNov/computational_metrics/pgd_nov_eval_resource_summary.csv,26.0,2.0,1840.964844,1035.0,43.807692,96.153846,8181.483448,pgd_nov_single_pass_eval,0.890244,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Original,BIM eval summary,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/bim_eval_resource_summary.csv,26.0,2.0,1835.156250,977.0,44.346154,96.153846,8161.414007,bim_single_pass_eval,0.890201,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Original,FGSM eval summary,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/fgsm_eval_resource_summary.csv,5.0,2.0,1834.390625,977.0,30.800000,80.000000,1132.489028,rfgsm_single_pass_eval,0.184965,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Original,PGD eval summary,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/pgd_eval_resource_summary.csv,26.0,2.0,1835.242188,977.0,44.807692,96.153846,8140.325283,pgd_single_pass_eval,0.890184,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 📌 Compact Table

,section,metric_name,student,attack_train_type,stage,epochs,learning_rate,iters,n_samples,max_cpu_ram_mb,max_gpu_mem_mb,avg_gpu_util_percent,gpu_active_percent_of_samples,est_gpu_energy_j,total_elapsed_minutes,estimated_macs,estimated_flops,parameter_count
0,Original,Base training summary,NaN,NaN,base_training,20.0,0.001,NaN,34.0,1788.425781,767.0,32.705882,97.058824,9141.918393,1.158622,NaN,NaN,NaN
1,Original,Base model profile,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6405760.0,12811520.0,266881.0
2,Original,FGSM student train summary,student_base_01,fgsm,NaN,5.0,0.001,NaN,52.0,1820.218750,977.0,45.269231,98.076923,16244.854219,1.764820,NaN,NaN,NaN
3,Original,FGSM student train summary,student_base_02,fgsm,NaN,5.0,0.001,NaN,42.0,1821.609375,977.0,43.738095,97.619048,14833.025604,1.777239,NaN,NaN,NaN
4,Original,FGSM student train summary,student_base_03,fgsm,NaN,5.0,0.001,NaN,53.0,1821.718750,977.0,44.566038,98.113208,15440.020339,1.798361,NaN,NaN,NaN
5,Original,FGSM student train summary,student_base_04,fgsm,NaN,5.0,0.001,NaN,43.0,1821.832031,977.0,44.186047,97.674419,14802.169894,1.818084,NaN,NaN,NaN
6,Original,BIM student train summary,student_base_01,bim,NaN,5.0,0.001,10.0,245.0,1822.207031,977.0,46.285714,99.591837,71795.128152,8.249693,NaN,NaN,NaN
7,Original,BIM student train summary,student_base_02,bim,NaN,5.0,0.001,10.0,242.0,1822.339844,977.0,46.714876,99.586777,70430.469181,8.148919,NaN,NaN,NaN
8,Original,BIM student train summary,student_base_03,bim,NaN,5.0,0.001,10.0,242.0,1822.441406,977.0,46.801653,99.586777,70519.363345,8.149116,NaN,NaN,NaN
9,Original,BIM student train summary,student_base_04,bim,NaN,5.0,0.001,10.0,243.0,1822.550781,977.0,46.679012,99.588477,70844.490129,8.182624,NaN,NaN,NaN


In [ ]:
!pip install xlsxwriter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 10.2 MB/s eta 0:00:00


In [ ]:


import os
import re
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

# -----------------------------
# base paths
# -----------------------------
BASE_PROJ_DIR = "/content/drive/MyDrive/298/Jena/proj_v3"
COMP_METRICS_DIR = os.path.join(BASE_PROJ_DIR, "computational_metrics")

JENANOV_DIR = os.path.join(BASE_PROJ_DIR, "jenaNov")
JENANOV_COMP_METRICS_DIR = os.path.join(JENANOV_DIR, "computational_metrics")

OUTPUT_XLSX = os.path.join(BASE_PROJ_DIR, "computational_metrics_report.xlsx")

if not os.path.exists(BASE_PROJ_DIR):
    raise FileNotFoundError(
        f"❌ Base directory not found:\n{BASE_PROJ_DIR}\n"
        "Make sure Google Drive is mounted and the path is correct."
    )

print("✅ Drive mounted and base directory found.")

# -----------------------------
# helper functions
# -----------------------------
def safe_read_csv(path):
    if os.path.exists(path):
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f"⚠️ Could not read {path}: {e}")
            return None
    else:
        print(f"⚠️ Missing file: {path}")
        return None

def add_source_columns(df, section, metric_name, source_path):
    if df is None or len(df) == 0:
        return None
    df = df.copy()
    df.insert(0, "section", section)
    df.insert(1, "metric_name", metric_name)
    df.insert(2, "source_csv", source_path)
    return df

def full_csv(path, section, metric_name):
    df = safe_read_csv(path)
    if df is None or len(df) == 0:
        return None
    return add_source_columns(df, section, metric_name, path)

def safe_sheet_name(name, used_names):
    # Excel sheet name max length = 31 and cannot contain: []:*?/\
    cleaned = re.sub(r'[\[\]\:\*\?\/\\]', '_', name)
    cleaned = cleaned[:31]
    original = cleaned
    i = 1
    while cleaned in used_names:
        suffix = f"_{i}"
        cleaned = original[:31-len(suffix)] + suffix
        i += 1
    used_names.add(cleaned)
    return cleaned

def auto_adjust_column_widths(writer, sheet_name, df):
    worksheet = writer.sheets[sheet_name]
    for idx, col in enumerate(df.columns):
        max_len = max(
            len(str(col)),
            *(len(str(x)) for x in df[col].astype(str).fillna(""))
        ) if len(df) > 0 else len(str(col))
        max_len = min(max_len + 2, 40)
        worksheet.set_column(idx, idx, max_len)

# -----------------------------
# file definitions
# -----------------------------
files_summary = [
    ("Base",  "Base training summary", os.path.join(COMP_METRICS_DIR, "base_resource_summary.csv")),
    ("Base",  "Base model profile",    os.path.join(COMP_METRICS_DIR, "base_model_profile.csv")),

    ("Original", "FGSM student train summary", os.path.join(COMP_METRICS_DIR, "fgsm_student_train_resource_summary.csv")),
    ("Original", "BIM student train summary",  os.path.join(COMP_METRICS_DIR, "bim_student_train_resource_summary.csv")),
    ("Original", "PGD student train summary",  os.path.join(COMP_METRICS_DIR, "pgd_student_train_resource_summary.csv")),

    ("Original", "FGSM transfer summary", os.path.join(COMP_METRICS_DIR, "fgsm_transfer_resource_summary.csv")),
    ("Original", "BIM transfer summary",  os.path.join(COMP_METRICS_DIR, "bim_transfer_resource_summary.csv")),
    ("Original", "PGD transfer summary",  os.path.join(COMP_METRICS_DIR, "pgd_transfer_resource_summary.csv")),

    ("Original", "FGSM eval summary", os.path.join(COMP_METRICS_DIR, "fgsm_eval_resource_summary.csv")),
    ("Original", "BIM eval summary",  os.path.join(COMP_METRICS_DIR, "bim_eval_resource_summary.csv")),
    ("Original", "PGD eval summary",  os.path.join(COMP_METRICS_DIR, "pgd_eval_resource_summary.csv")),

    ("Novel", "Model sizes", os.path.join(JENANOV_DIR, "jenaNov_model_sizes.csv")),

    ("Novel", "FGSM NOV student train summary", os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_student_train_resource_summary.csv")),
    ("Novel", "BIM NOV student train summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_student_train_resource_summary.csv")),
    ("Novel", "PGD NOV student train summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_student_train_resource_summary.csv")),

    ("Novel", "FGSM NOV transfer summary", os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_transfer_resource_summary.csv")),
    ("Novel", "BIM NOV transfer summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_transfer_resource_summary.csv")),
    ("Novel", "PGD NOV transfer summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_transfer_resource_summary.csv")),

    ("Novel", "FGSM NOV eval summary", os.path.join(JENANOV_COMP_METRICS_DIR, "fgsm_nov_eval_resource_summary.csv")),
    ("Novel", "BIM NOV eval summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "bim_nov_eval_resource_summary.csv")),
    ("Novel", "PGD NOV eval summary",  os.path.join(JENANOV_COMP_METRICS_DIR, "pgd_nov_eval_resource_summary.csv")),
]

# -----------------------------
# load all dataframes
# -----------------------------
loaded = {}
all_tables = []

for section, metric_name, path in files_summary:
    df = full_csv(path, section, metric_name)
    if df is not None:
        loaded[(section, metric_name)] = df
        all_tables.append(df)

combined = pd.concat(all_tables, ignore_index=True) if all_tables else pd.DataFrame()

# grouped views
base_model_df = combined[
    combined["metric_name"].str.contains("Base training summary|Base model profile|Model sizes", case=False, na=False)
].copy()

training_df = combined[
    combined["metric_name"].str.contains("train summary", case=False, na=False)
].copy()

transfer_df = combined[
    combined["metric_name"].str.contains("transfer summary", case=False, na=False)
].copy()

evaluation_df = combined[
    combined["metric_name"].str.contains("eval summary", case=False, na=False)
].copy()

compact_cols = [
    "section", "metric_name", "student", "attack_train_type", "stage",
    "epochs", "learning_rate", "iters", "n_samples",
    "max_cpu_ram_mb", "max_gpu_mem_mb",
    "avg_gpu_util_percent", "gpu_active_percent_of_samples",
    "est_gpu_energy_j", "total_elapsed_minutes",
    "estimated_macs", "estimated_flops", "parameter_count"
]
compact_cols = [c for c in compact_cols if c in combined.columns]
compact_df = combined[compact_cols].copy() if len(combined) > 0 else pd.DataFrame()

# -----------------------------
# export to Excel
# -----------------------------
if len(combined) == 0:
    raise ValueError("❌ No computational metrics CSVs were loaded.")

used_names = set()

with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
    # Main summary sheets
    summary_sheets = {
        "All_Combined": combined,
        "Compact_View": compact_df,
        "Base_and_Model": base_model_df,
        "Training": training_df,
        "Transfer": transfer_df,
        "Evaluation": evaluation_df,
    }

    for sheet_name, df in summary_sheets.items():
        if df is not None and len(df) > 0:
            df.to_excel(writer, sheet_name=sheet_name, index=False)
            auto_adjust_column_widths(writer, sheet_name, df)
            writer.sheets[sheet_name].freeze_panes(1, 0)

    # Individual source sheets
    for (section, metric_name), df in loaded.items():
        sheet_name = safe_sheet_name(f"{section}_{metric_name}", used_names)
        df.to_excel(writer, sheet_name=sheet_name, index=False)
        auto_adjust_column_widths(writer, sheet_name, df)
        writer.sheets[sheet_name].freeze_panes(1, 0)

print(f"✅ Excel workbook saved to:\n{OUTPUT_XLSX}")

# Optional: preview main sheets in notebook
print("\nPreview: All Combined")
display(combined.head())

print("\nPreview: Compact View")
display(compact_df.head())

✅ Drive mounted and base directory found.
✅ Excel workbook saved to:
/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics_report.xlsx

Preview: All Combined


,section,metric_name,source_csv,n_samples,sample_interval_sec,max_cpu_ram_mb,max_gpu_mem_mb,avg_gpu_util_percent,gpu_active_percent_of_samples,est_gpu_energy_j,stage,total_elapsed_minutes,epochs,learning_rate,model,estimated_macs,estimated_flops,parameter_count,student,attack_train_type,iters,student_idx,model_name,path,params,size_mb
0,Base,Base training summary,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/base_resource_summary.csv,34.0,2.0,1788.425781,767.0,32.705882,97.058824,9141.918393,base_training,1.158622,20.0,0.001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Base,Base model profile,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/base_model_profile.csv,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,jena_base_transformer,6405760.0,12811520.0,266881.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Original,FGSM student train summary,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/fgsm_student_train_resource_summary.csv,52.0,2.0,1820.218750,977.0,45.269231,98.076923,16244.854219,NaN,1.764820,5.0,0.001,NaN,NaN,NaN,NaN,student_base_01,fgsm,NaN,NaN,NaN,NaN,NaN,NaN
3,Original,FGSM student train summary,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/fgsm_student_train_resource_summary.csv,42.0,2.0,1821.609375,977.0,43.738095,97.619048,14833.025604,NaN,1.777239,5.0,0.001,NaN,NaN,NaN,NaN,student_base_02,fgsm,NaN,NaN,NaN,NaN,NaN,NaN
4,Original,FGSM student train summary,/content/drive/MyDrive/298/Jena/proj_v3/computational_metrics/fgsm_student_train_resource_summary.csv,53.0,2.0,1821.718750,977.0,44.566038,98.113208,15440.020339,NaN,1.798361,5.0,0.001,NaN,NaN,NaN,NaN,student_base_03,fgsm,NaN,NaN,NaN,NaN,NaN,NaN



Preview: Compact View


,section,metric_name,student,attack_train_type,stage,epochs,learning_rate,iters,n_samples,max_cpu_ram_mb,max_gpu_mem_mb,avg_gpu_util_percent,gpu_active_percent_of_samples,est_gpu_energy_j,total_elapsed_minutes,estimated_macs,estimated_flops,parameter_count
0,Base,Base training summary,NaN,NaN,base_training,20.0,0.001,NaN,34.0,1788.425781,767.0,32.705882,97.058824,9141.918393,1.158622,NaN,NaN,NaN
1,Base,Base model profile,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6405760.0,12811520.0,266881.0
2,Original,FGSM student train summary,student_base_01,fgsm,NaN,5.0,0.001,NaN,52.0,1820.218750,977.0,45.269231,98.076923,16244.854219,1.764820,NaN,NaN,NaN
3,Original,FGSM student train summary,student_base_02,fgsm,NaN,5.0,0.001,NaN,42.0,1821.609375,977.0,43.738095,97.619048,14833.025604,1.777239,NaN,NaN,NaN
4,Original,FGSM student train summary,student_base_03,fgsm,NaN,5.0,0.001,NaN,53.0,1821.718750,977.0,44.566038,98.113208,15440.020339,1.798361,NaN,NaN,NaN
